# URA Hackathon — Hybrid PaddleOCR + VietOCR Rescue + Optional ResNet Gate

Notebook này mở rộng bản final no-LR theo hướng an toàn:

1. **PaddleOCR GPU** vẫn là OCR base.
2. **VietOCR** chỉ chạy như rescue khi Paddle blank/yếu, không thay toàn bộ OCR.
3. `product_name` vẫn dùng **rule/fuzzy no-LR** để tránh LogisticRegression hallucination.
4. **ResNet visual gate** được thêm dưới dạng optional switch, dùng để chặn product mâu thuẫn domain hình ảnh; mặc định tắt để pipeline vẫn ổn định.
5. Output chính: `/kaggle/working/submission.csv`.



## 1) Install PaddleOCR GPU + VietOCR

Cell này cài PaddleOCR GPU cho OCR base và cài VietOCR với **CPU-only Torch** để tránh lỗi Torch CUDA/NCCL trên Kaggle.

Nếu chỉ muốn chạy bản PaddleOCR no-LR không VietOCR, có thể bỏ qua phần VietOCR, nhưng notebook hybrid này cần `vietocr`.



In [1]:
# ============================================================
# Install PaddleOCR GPU + VietOCR for Kaggle
# ============================================================
import os, sys, subprocess, textwrap

PADDLE_VERSION = "3.3.0"
PADDLE_CUDA = "cu118"  # Kaggle-safe default. Option: "cu126" if your runtime supports CUDA 12.6.

print("Python:", sys.version)
print("Installing PaddlePaddle GPU:", PADDLE_VERSION, PADDLE_CUDA)
subprocess.run("nvidia-smi", shell=True)

# Clean old/conflicting installs.
subprocess.run(
    f"{sys.executable} -m pip uninstall -y "
    f"paddlepaddle paddlepaddle-gpu paddleocr paddlex torch torchvision torchaudio vietocr",
    shell=True,
    check=False,
)

# Paddle GPU wheel.
subprocess.run(
    f"{sys.executable} -m pip install -q --no-cache-dir "
    f"paddlepaddle-gpu=={PADDLE_VERSION} "
    f"-i https://www.paddlepaddle.org.cn/packages/stable/{PADDLE_CUDA}/",
    shell=True,
    check=True,
)

# CPU-only Torch for VietOCR. This avoids libtorch_cuda/NCCL conflicts.
subprocess.run(
    f"{sys.executable} -m pip install -q --no-cache-dir "
    f"torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu",
    shell=True,
    check=True,
)

# OCR + baseline dependencies.
subprocess.run(
    f"{sys.executable} -m pip install -q --no-cache-dir "
    f"paddleocr vietocr scikit-learn tqdm pillow opencv-python-headless",
    shell=True,
    check=True,
)

# Re-pin Paddle GPU after dependency install, in case dependency resolver touched it.
subprocess.run(
    f"{sys.executable} -m pip install -q --no-cache-dir --force-reinstall --no-deps "
    f"paddlepaddle-gpu=={PADDLE_VERSION} "
    f"-i https://www.paddlepaddle.org.cn/packages/stable/{PADDLE_CUDA}/",
    shell=True,
    check=True,
)

print("Install done. If import fails in next cell, restart the Kaggle session once, then run from Cell 2.")



Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Installing PaddlePaddle GPU: 3.3.0 cu118
Thu Jun 18 12:04:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                       

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 GB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 54.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 247.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 213.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 307.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 248.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 313.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 335.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 173.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 190.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 361.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 401.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 GB 17.9 MB/s eta 0:00:00
Install done. If import fails in next cell, restart the Kaggle session once, then run from Cell 2.


## 2) Version check + GPU check



In [1]:
import os, sys, subprocess, platform, importlib.util as _importlib_util

# Keep Paddle logs less noisy.
os.environ.setdefault("GLOG_minloglevel", "2")
os.environ.setdefault("FLAGS_allocator_strategy", "auto_growth")

import paddle


def safe_import_paddleocr(hide_torch: bool = True):
    """
    Kaggle fix:
    PaddleOCR 3.x imports PaddleX -> ModelScope. ModelScope probes torch during import.
    Some Kaggle runtimes have a Torch/NCCL mismatch after Paddle install, causing:
        ImportError: libtorch_cuda.so: undefined symbol: ncclCommShrink

    General OCR with Paddle backend does not need torch, so during PaddleOCR import only,
    we hide torch from importlib.util.find_spec(). This prevents ModelScope logger from
    importing the broken torch package.
    """
    if "paddleocr" in sys.modules:
        import paddleocr
        from paddleocr import PaddleOCR
        return paddleocr, PaddleOCR

    if not hide_torch:
        import paddleocr
        from paddleocr import PaddleOCR
        return paddleocr, PaddleOCR

    original_find_spec = _importlib_util.find_spec

    def patched_find_spec(name, *args, **kwargs):
        if name == "torch" or name.startswith("torch."):
            return None
        return original_find_spec(name, *args, **kwargs)

    _importlib_util.find_spec = patched_find_spec
    try:
        import paddleocr
        from paddleocr import PaddleOCR
        return paddleocr, PaddleOCR
    finally:
        _importlib_util.find_spec = original_find_spec


paddleocr, PaddleOCR = safe_import_paddleocr(hide_torch=True)

print("Python        :", sys.version.split()[0])
print("Platform      :", platform.platform())
print("paddle        :", paddle.__version__)
print("paddleocr     :", getattr(paddleocr, "__version__", "unknown"))
print("CUDA compiled :", paddle.device.is_compiled_with_cuda())

try:
    print("CUDA devices  :", paddle.device.cuda.device_count())
except Exception as e:
    print("CUDA devices  : check failed", repr(e))

try:
    paddle.utils.run_check()
except Exception as e:
    print("paddle run_check failed:", repr(e))
    print("If this is CUDA-version related, rerun install cell with PADDLE_CUDA='cu126' or restart session.")



/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Python        : 3.12.13
Platform      : Linux-6.12.90+-x86_64-with-glibc2.35
paddle        : 3.3.0
paddleocr     : 3.7.0
CUDA compiled : True
CUDA devices  : 2
Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
[2026-06-18 12:13:48,429] [    INFO] cloud_utils.py:135 - get cluster from args:job_server:None pods:['rank:0 id:None addr:127.0.0.1 port:None visible_gpu:[] trainers:["gpu:[\'0\'] endpoint:127.0.0.1:37627 rank:0", "gpu:[\'1\'] endpoint:127.0.0.1:33679 rank:1"]'] job_stage_flag:None hdfs:None


PaddlePaddle works well on 1 GPU.


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
======================= Modified FLAGS detected =======================
FLAGS(name='FLAGS_cupti_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/cuda_cupti/lib', default_value='')
FLAGS(name='FLAGS_cublas_dir', current_value='/usr/local/lib/python3.12/dist-packages/paddle/../nvidia/cublas/lib', default_value='')
FLAGS(name=

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


## 3) Load data + auto-detect images



In [5]:
import re, time, zipfile, csv, json, os
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

# ------------------------------------------------------------
# Dataset discovery
# ------------------------------------------------------------
IMAGE_DIR_CANDIDATES = (
    "test_images/test_images/images",  # common Kaggle extracted nested layout
    "test_images/images",
    "test_images",
    "images",
    "test/images",
    "test/test_images/images",
    "test/test_images",
)
IMAGE_ZIP_NAMES = (
    "test_images.zip",
    "images.zip",
    "test/test_images.zip",
    "test/images.zip",
)


def _input_roots() -> list[Path]:
    roots: list[Path] = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        comp_root = kaggle_input / "competitions"
        if comp_root.exists():
            roots.extend(sorted(p for p in comp_root.iterdir() if p.is_dir()))
        roots.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))
    for local in [Path("public_dataset"), Path("smce_dataset/test"), Path(".")]:
        if local.exists():
            roots.append(local.resolve())
    seen = set()
    out = []
    for r in roots:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


def resolve_input_dir() -> Path:
    for root in _input_roots():
        if (root / "test.csv").exists():
            return root
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in sorted(kaggle_input.rglob("test.csv")):
            return p.parent
    raise FileNotFoundError(
        "Không tìm thấy test.csv. Trên Kaggle: Add Input → Competition Data → The 2nd URA Hackathon."
    )


def _find_images_dir(input_dir: Path) -> Path | None:
    for rel in IMAGE_DIR_CANDIDATES:
        p = input_dir / rel
        if p.is_dir() and any(p.glob("*.jpg")):
            return p
    # fallback: find any folder with many jpg files under input_dir
    jpg_dirs = []
    for jpg in input_dir.rglob("*.jpg"):
        jpg_dirs.append(jpg.parent)
        if len(jpg_dirs) >= 30:
            break
    if jpg_dirs:
        # choose most common parent among first jpgs
        from collections import Counter
        return Counter(jpg_dirs).most_common(1)[0][0]
    return None


def _find_images_zip(input_dir: Path) -> Path | None:
    for rel in IMAGE_ZIP_NAMES:
        p = input_dir / rel
        if p.is_file():
            return p
    for p in sorted(input_dir.rglob("*.zip")):
        name = p.name.lower()
        if "train" in name and "test" not in name:
            continue
        if "image" in name or "test" in name:
            return p
    return None


def setup_images_dir(input_dir: Path, work_dir: Path) -> Path:
    images_dir = _find_images_dir(input_dir)
    if images_dir is not None:
        return images_dir

    zip_path = _find_images_zip(input_dir)
    if zip_path is None:
        raise FileNotFoundError(f"Không tìm thấy test_images ở {input_dir}")

    extract_root = work_dir / "dataset_images"
    extract_root.mkdir(parents=True, exist_ok=True)
    if not any(extract_root.rglob("*.jpg")):
        print(f"Extracting {zip_path} -> {extract_root}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_root)

    images_dir = _find_images_dir(extract_root)
    if images_dir is None:
        raise FileNotFoundError(f"Extracted zip nhưng không thấy jpg trong {extract_root}")
    return images_dir


INPUT_DIR = resolve_input_dir()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
IMAGES_DIR = setup_images_dir(INPUT_DIR, WORK_DIR)

TEST_CSV = INPUT_DIR / "test.csv"
SAMPLE_CSV = INPUT_DIR / "sample_submission.csv"
OUTPUT_CSV = WORK_DIR / "submission.csv"
CHECKPOINT_CSV = WORK_DIR / "checkpoint_hybrid_paddle_vietocr_resnet.csv"

if not SAMPLE_CSV.exists():
    raise FileNotFoundError(f"Không tìm thấy sample_submission.csv ở {INPUT_DIR}")

test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
sample_df = pd.read_csv(SAMPLE_CSV, keep_default_na=False)

test_df["image_id"] = test_df["image_id"].astype(str).str.strip()
sample_df["image_id"] = sample_df["image_id"].astype(str).str.strip()

image_ids_on_disk = {p.stem for p in IMAGES_DIR.glob("*.jpg")}

train_labels_df = None
for labels_path in [
    INPUT_DIR / "train_labels.csv",
    INPUT_DIR / "train.csv",
    Path("/kaggle/input").joinpath("train_labels.csv"),
    Path("public_dataset/train_labels.csv"),
]:
    if labels_path.exists():
        train_labels_df = pd.read_csv(labels_path, keep_default_na=False)
        break

print(f"Input dir   : {INPUT_DIR}")
print(f"Images dir  : {IMAGES_DIR}")
print(f"Working dir : {WORK_DIR}")
print(f"Test rows   : {len(test_df):,}")
print(f"Sample rows : {len(sample_df):,}")
print(f"JPG files   : {len(image_ids_on_disk):,}")

missing = set(test_df["image_id"]) - image_ids_on_disk
if missing:
    print(f"WARNING: {len(missing):,} image_id không có file jpg. Ví dụ:", list(sorted(missing))[:5])
else:
    print("All test image_ids have matching jpg files.")

if train_labels_df is not None:
    print(f"Train labels: {len(train_labels_df):,} rows from {labels_path}")
    print("Columns     :", list(train_labels_df.columns))
else:
    print("Train labels: not found — product_name sẽ dùng rules-only.")

display(test_df.head(3))



Input dir   : /kaggle/input/competitions/the-2nd-ura-hackathon
Images dir  : /kaggle/input/competitions/the-2nd-ura-hackathon/test_images/test_images/images
Working dir : /kaggle/working
Test rows   : 2,006
Sample rows : 2,006
JPG files   : 2,006
All test image_ids have matching jpg files.
Train labels: 4,892 rows from /kaggle/input/competitions/the-2nd-ura-hackathon/train_labels.csv
Columns     : ['image_id', 'ocr_text', 'product_name']


,image_id
0,img_2934
1,img_2935
2,img_2936


## 4) Product rules



In [6]:
# BRAND_RULES: (regex, canonical_name, [product_line_keywords])
import re
import unicodedata
BRAND_RULES = [
    # ============================================================
    # 1) HA LONG / CANFOCO / DO HOP / PATE - VERY SPECIFIC FIRST
    # ============================================================

    # Ha Long Canfoco + Pate Cot Den
    (
        r"(ha\s*long|halong|h\s*long|h[aạàáảã]\s*long|ha\s*ong).{0,40}"
        r"(canfoco|can\s*foco|c[aăâ]n\s*f[o0]c[o0]|nfoco|halcn\s*foco).{0,60}"
        r"(pate|pat[eê]|pa\s*te).{0,40}"
        r"(c[ộo]t\s*đ[èe]n|cot\s*den)",
        "Ha Long Canfoco Pate Cột Đèn",
        [],
    ),
    (
        r"(pate|pat[eê]|pa\s*te).{0,40}"
        r"(c[ộo]t\s*đ[èe]n|cot\s*den).{0,80}"
        r"(ha\s*long|halong|canfoco|can\s*foco)",
        "Ha Long Canfoco Pate Cột Đèn",
        [],
    ),

    # Ha Long Canfoco + Pate
    (
        r"(ha\s*long|halong|h\s*long|ha\s*ong).{0,40}"
        r"(canfoco|can\s*foco|c[aăâ]n\s*f[o0]c[o0]|nfoco|canfood).{0,60}"
        r"(pate|pat[eê]|pa\s*te)",
        "Ha Long Canfoco Pate",
        [],
    ),
    (
        r"(canfoco|can\s*foco|c[aăâ]n\s*f[o0]c[o0]|nfoco|canfood).{0,60}"
        r"(pate|pat[eê]|pa\s*te)",
        "Ha Long Canfoco Pate",
        [],
    ),

    # Ha Long Canfoco generic
    (
        r"(ha\s*long|halong|h\s*long|h[aạàáảã]\s*long|ha\s*ong).{0,50}"
        r"(canfoco|can\s*foco|c[aăâ]n\s*f[o0]c[o0]|nfoco|canfood|halcn\s*foco)",
        "Ha Long Canfoco",
        [],
    ),
    (
        r"(canfoco|can\s*foco|c[aăâ]n\s*f[o0]c[o0]|nfoco|canfood|halcn\s*foco)",
        "Ha Long Canfoco",
        [],
    ),

    # Do Hop Ha Long / Cong ty Do Hop Ha Long
    (
        r"(c[oôơ]ng\s*ty|cty).{0,40}"
        r"(đ[ồo]\s*h[ộo]p|do\s*hop|d[oôơ]\s*h[oôơ]p).{0,40}"
        r"(h[ạa]\s*long|ha\s*long|halong|h\s*long|ha\s*ong)",
        "Đồ Hộp Hạ Long",
        [],
    ),
    (
        r"(đ[ồo]\s*h[ộo]p|do\s*hop|d[oôơ]\s*h[oôơ]p).{0,40}"
        r"(h[ạa]\s*long|ha\s*long|halong|h\s*long|ha\s*ong)",
        "Đồ Hộp Hạ Long",
        [],
    ),

    # Pate Cot Den Hai Phong
    (
        r"(pate|pat[eê]|pa\s*te).{0,40}"
        r"(c[ộo]t\s*đ[èe]n|cot\s*den|c[o0]t\s*d[e3]n).{0,50}"
        r"(h[ảa]i\s*ph[òo]ng|hai\s*phong)?",
        "Pate Cột Đèn Hải Phòng",
        [],
    ),
    (
        r"(c[ộo]t\s*đ[èe]n|cot\s*den|c[o0]t\s*d[e3]n).{0,50}"
        r"(h[ảa]i\s*ph[òo]ng|hai\s*phong)",
        "Pate Cột Đèn Hải Phòng",
        [],
    ),

    # Hạ Long Pate generic
    (
        r"(h[ạa]\s*long|ha\s*long|halong).{0,40}(pate|pat[eê]|pa\s*te)",
        "Ha Long Canfoco Pate",
        [],
    ),
    (
        r"(pate|pat[eê]|pa\s*te).{0,40}(h[ạa]\s*long|ha\s*long|halong)",
        "Ha Long Canfoco Pate",
        [],
    ),

    # ============================================================
    # 2) OTHER CANNED MEAT / PATE / SAUSAGE BRANDS
    # ============================================================

    (
        r"vissan",
        "Vissan",
        [
            "pate heo",
            "pate gan",
            "pate gà",
            "pate ga",
            "xúc xích",
            "xuc xich",
            "lạp xưởng",
            "lap xuong",
            "thịt hộp",
            "thit hop",
        ],
    ),
    (
        r"(cp|c\.p\.|c p)\b",
        "CP",
        [
            "pate",
            "xúc xích",
            "xuc xich",
            "thịt hộp",
            "thit hop",
            "heo hai lát",
            "pork luncheon meat",
        ],
    ),
    (
        r"(pate\s*gan|pat[eê]\s*gan)",
        "Pate Gan",
        [],
    ),
    (
        r"(th[ịi]t\s*h[ộo]p|thit\s*hop|luncheon\s*meat|pork\s*luncheon)",
        "Thịt hộp",
        [],
    ),
    (
        r"(x[úu]c\s*x[íi]ch|xuc\s*xich|sausage)",
        "Xúc xích",
        [],
    ),
    (
        r"(l[ạa]p\s*x[ưu]ởng|lap\s*xuong)",
        "Lạp xưởng",
        [],
    ),
    (
        r"(hafi)\b",
        "Hafi",
        ["pate"],
    ),
    (
        r"(ba\s*huan|ba\s*huân)",
        "Ba Huân",
        ["pate"],
    ),
    (
        r"(san\s*ha|san\s*hà)",
        "San Hà",
        ["pate"],
    ),
    (
        r"(long\s*bien|long\s*biên)",
        "Long Biên",
        ["pate"],
    ),

    # Canned fish / sardine
    (
        r"(quang\s*h[oô]ng|quanghong).{0,40}(sardine|c[áa]\s*m[òo]i|ca\s*moi)",
        "QUANG HONG SARDINE",
        [],
    ),
    (
        r"(sardine|c[áa]\s*m[òo]i|ca\s*moi|c[áa]\s*h[ộo]p|ca\s*hop)",
        "Sardine",
        [],
    ),
    (
        r"(three\s*lady\s*cooks|3\s*lady\s*cooks|ba\s*c[oô]\s*g[aà])",
        "Three Lady Cooks",
        [],
    ),
    (
        r"\bligo\b",
        "Ligo",
        ["sardines", "sardine"],
    ),
    (
        r"\brosa\b",
        "Rosa",
        ["sardines", "sardine"],
    ),

    # Generic pate - keep late inside canned section
    (
        r"\b(pate|pat[eê]|pa\s*te)\b",
        "Pate",
        [],
    ),

    # ============================================================
    # 3) HIGHLANDS / COFFEE / TEA
    # ============================================================

    # Specific Highlands drinks first
    (
        r"(highlands|higlands|highland|hightlands|higland).{0,80}"
        r"(tr[àa]\s*sen\s*v[àa]ng).{0,60}(tr[àa]\s*v[ảa]i)",
        "Highlands Coffee trà sen vàng, trà vải",
        [],
    ),
    (
        r"(tr[àa]\s*sen\s*v[àa]ng).{0,60}(tr[àa]\s*v[ảa]i).{0,80}"
        r"(highlands|highland)",
        "Highlands Coffee trà sen vàng, trà vải",
        [],
    ),
    (
        r"(highlands|highland).{0,80}(tr[àa]\s*sen\s*v[àa]ng)",
        "Trà Sen Vàng Highlands Coffee",
        [],
    ),
    (
        r"(highlands|highland).{0,80}(tr[àa]\s*v[ảa]i)",
        "Trà Vải Highlands Coffee",
        [],
    ),
    (
        r"(highlands|highland).{0,80}(b[áa]nh\s*m[ìi]|banh\s*mi)",
        "BÁNH MÌ HIGHLANDS",
        [],
    ),
    (
        r"(tr[àa]\s*ph[úu]c\s*ki[ếe]n).{0,80}(americano\s*v[ảa]i)",
        "TRÀ PHÚC KIẾN SEN VẢI & AMERICANO VẢI",
        [],
    ),
    (
        r"(highlands|higlands|highland|hightlands|higland|hlds).{0,40}(coffee|cofee|cafe|cà\s*phê|cf)",
        "HIGHLANDS COFFEE",
        [],
    ),
    (
        r"(coffee|cofee|cafe|cà\s*phê|cf).{0,40}(highlands|highland)",
        "HIGHLANDS COFFEE",
        [],
    ),

    # Other coffee brands
    (
        r"(the\s*coffee\s*house|coffee\s*house|\btch\b)",
        "The Coffee House",
        ["trà đào", "trà vải", "cà phê sữa", "cold brew"],
    ),
    (
        r"(ph[úu]c\s*long|phuc\s*long)",
        "Phúc Long",
        ["trà sữa", "trà đào", "trà vải", "cà phê"],
    ),
    (
        r"(trung\s*nguy[eê]n|g7\s*coffee|\bg7\b)",
        "Trung Nguyên G7",
        ["coffee", "cà phê"],
    ),
    (
        r"(nescafe|nescaf[eé]|n[eé]scafe)",
        "Nescafé",
        ["cafe sữa", "cà phê sữa", "3 in 1", "2 in 1"],
    ),
    (
        r"(coffee\s*mate|coffeemate)",
        "Nestlé Coffee Mate",
        [],
    ),

    # Generic coffee - keep after brand rules
    (
        r"\b(coffee|cofee|cafe|cà\s*phê|cf)\b",
        "Coffee",
        [],
    ),

    # ============================================================
    # 4) NESTLE / NAN / APTAMIL / ABBOTT / INFANT FORMULA
    # ============================================================

    # Nestlé NAN very specific
    (
        r"(nestle|nestl[eé]).{0,40}(nan).{0,40}(opti\s*pro\s*plus|optipro\s*plus|opti\s*proplus|optiproplus)",
        "Nestlé NAN OPTI pro plus",
        [],
    ),
    (
        r"(nan).{0,40}(opti\s*pro\s*plus|optipro\s*plus|opti\s*proplus|optiproplus)",
        "Nestlé NAN OPTI pro plus",
        [],
    ),
    (
        r"(nestle|nestl[eé]).{0,40}(nan).{0,40}(opti\s*pro|optipro)",
        "Nestlé NAN OPTIpro",
        [],
    ),
    (
        r"(nan).{0,40}(opti\s*pro|optipro)",
        "Nestlé NAN OPTIpro",
        [],
    ),
    (
        r"(nestle|nestl[eé]).{0,40}(nan).{0,40}(infinipro|infini\s*pro|a2)",
        "Sữa bột Nestlé NAN Infinipro A2",
        [],
    ),
    (
        r"(nan).{0,40}(infinipro|infini\s*pro|a2)",
        "Sữa bột Nestlé NAN Infinipro A2",
        [],
    ),
    (
        r"(nestle|nestl[eé]).{0,40}(nan).{0,40}(supreme\s*pro)",
        "Nestlé Nan Supreme Pro",
        [],
    ),
    (
        r"(nan).{0,40}(supreme\s*pro)",
        "Nestlé Nan Supreme Pro",
        [],
    ),
    (
        r"(nestle|nestl[eé]).{0,40}(alfamino|alfa\s*mino)",
        "NESTLÉ Alfamino INFANT",
        [],
    ),
    (
        r"(alfamino|alfa\s*mino)",
        "NESTLÉ Alfamino INFANT",
        [],
    ),
    (
        r"(nestle|nestl[eé]).{0,40}\bnan\b",
        "Nestlé Nan",
        [],
    ),
    (
        r"\bnan\b",
        "Nan",
        [],
    ),

    # Aptamil
    (
        r"(aptamil).{0,40}(profutura|pro\s*futura)",
        "Aptamil Profutura",
        [],
    ),
    (
        r"(aptamil).{0,40}(essensis)",
        "Aptamil Essensis",
        [],
    ),
    (
        r"(aptamil).{0,40}(pronutra|pro\s*nutra)",
        "Aptamil Pronutra",
        [],
    ),
    (
        r"aptamil",
        "Aptamil",
        [],
    ),

    # Abbott
    (
        r"(ensure).{0,40}(gold)",
        "Abbott Ensure Gold",
        [],
    ),
    (
        r"\bensure\b",
        "Abbott Ensure",
        ["gold", "original", "plus"],
    ),
    (
        r"(pediasure|pedia\s*sure).{0,40}(gold)",
        "Abbott PediaSure Gold",
        [],
    ),
    (
        r"(pediasure|pedia\s*sure)",
        "Abbott PediaSure",
        [],
    ),
    (
        r"similac",
        "Abbott Similac",
        [],
    ),
    (
        r"glucerna",
        "Abbott Glucerna",
        [],
    ),

    # Other formula/milk powder
    (
        r"friso\b",
        "Friso",
        ["gold", "comfort", "prestige"],
    ),
    (
        r"meiji\b",
        "Meiji",
        ["growing", "step"],
    ),
    (
        r"(morinaga|chilmil|hagukumi)",
        "Morinaga",
        [],
    ),
    (
        r"(enfa|enfamil|enfagrow|enfamil\s*a\+)",
        "Enfa",
        ["a+", "enfagrow", "enfamil"],
    ),
    (
        r"(illuma|s26|s-26|wyeth)",
        "Wyeth",
        ["illuma", "s26"],
    ),

    # Generic Nestle after specific NAN/Milo/Nescafe
    (
        r"(milo)\b",
        "Nestlé Milo",
        [],
    ),
    (
        r"(nestle|nestl[eé])",
        "Nestlé",
        [
            "milo",
            "coffee mate",
            "carnation",
            "nestea",
            "nan",
            "sữa bột",
            "sua bot",
        ],
    ),

    # ============================================================
    # 5) VINAMILK / VIETNAMESE DAIRY
    # ============================================================

    (
        r"(vinamilk).{0,50}(flex)",
        "Vinamilk Flex",
        [],
    ),
    (
        r"(vinamilk).{0,50}(adm\s*gold|a\s*d\s*m\s*gold)",
        "Vinamilk ADM Gold",
        [],
    ),
    (
        r"(vinamilk).{0,50}(sure\s*prevent|sure)",
        "Vinamilk Sure",
        [],
    ),
    (
        r"(vinamilk).{0,50}(canxi|calci|calcium)",
        "Vinamilk Canxi",
        [],
    ),
    (
        r"(vinamilk).{0,50}(dielac)",
        "Vinamilk Dielac",
        [],
    ),
    (
        r"(vinamilk).{0,50}(grow)",
        "Vinamilk Grow",
        [],
    ),
    (
        r"(vinamilk).{0,50}(colos\s*baby|colosbaby)",
        "Vinamilk ColosBaby",
        [],
    ),
    (
        r"(vinamilk).{0,50}([ôo]ng\s*th[ọo]|ong\s*tho)",
        "Vinamilk Ông Thọ",
        [],
    ),
    (
        r"vinamilk",
        "Vinamilk",
        [
            "flex",
            "adm gold",
            "sure",
            "canxi",
            "colosbaby",
            "colos baby",
            "ông thọ",
            "ong tho",
            "dielac",
            "grow",
        ],
    ),

    # TH True Milk
    (
        r"(th\s*true|thtrue).{0,50}(milk|s[ữu]a)",
        "TH True Milk",
        [],
    ),
    (
        r"(th\s*true|thtrue).{0,50}(yogurt|s[ữu]a\s*chua)",
        "TH True Yogurt",
        [],
    ),
    (
        r"(th\s*true|thtrue)",
        "TH True Milk",
        ["true yogurt", "grow", "school milk", "true butter"],
    ),

    # Dutch Lady / Co Gai Ha Lan
    (
        r"(dutch\s*lady|c[ôo]\s*g[áa]i\s*h[àa]\s*lan|co\s*gai\s*ha\s*lan)",
        "Dutch Lady",
        ["grow", "complete", "canxi", "mom"],
    ),

    # Nutifood
    (
        r"(nutifood|nuti\s*food|nuti\b).{0,50}(grow\s*plus|growplus)",
        "Nutifood Grow PLUS",
        [],
    ),
    (
        r"(nutifood|nuti\s*food|nuti\b).{0,50}(pedia)",
        "Nutifood Pedia",
        [],
    ),
    (
        r"(nutifood|nuti\s*food|nuti\b)",
        "Nutifood",
        ["growplus", "grow plus", "pedia", "iq"],
    ),

    # Other dairy
    (
        r"(ba\s*vi|bavi|ba\s*vì)",
        "Ba Vì",
        ["gold"],
    ),
    (
        r"lothamilk",
        "Lothamilk",
        ["canxi"],
    ),
    (
        r"yomost",
        "Yomost",
        [],
    ),
    (
        r"(dalat\s*milk|đà\s*lạt\s*milk|da\s*lat\s*milk)",
        "Đà Lạt Milk",
        [],
    ),
    (
        r"(kun\b|kun\s*milk)",
        "Kun",
        ["chocolate", "strawberry"],
    ),
    (
        r"fami\b",
        "Fami",
        ["canxi", "kid"],
    ),
    (
        r"anlene",
        "Anlene",
        ["gold", "concentrate"],
    ),
    (
        r"anchor\b",
        "Anchor",
        ["butter", "cream"],
    ),
    (
        r"(m[ộo]c\s*ch[âa]u|moc\s*chau)",
        "Mộc Châu Milk",
        [],
    ),
    (
        r"(love'?in\s*farm|lovein\s*farm)",
        "Love'in Farm",
        [],
    ),
    (
        r"(lif|lif\s*kun)",
        "LIF",
        [],
    ),

    # Generic milk - keep very late
    (
        r"(s[ữu]a\s*b[ộo]t|sua\s*bot|milk\s*powder)",
        "Sữa bột",
        [],
    ),
    (
        r"(s[ữu]a\s*t[ưu]ơi|sua\s*tuoi|fresh\s*milk)",
        "Sữa tươi",
        [],
    ),

    # ============================================================
    # 6) SKINCARE / BEAUTY / OTHER PRODUCTS OBSERVED
    # ============================================================

    (
        r"(acnes).{0,50}(vitamin\s*cleanser|cleanser)",
        "Acnes Vitamin Cleanser",
        [],
    ),
    (
        r"\bacnes\b",
        "Acnes",
        ["vitamin cleanser", "cleanser", "gel", "cream"],
    ),
    (
        r"(hanayuki|hana\s*yuki)",
        "Hanayuki",
        [],
    ),
    (
        r"(mailisa)",
        "Mailisa",
        [],
    ),
    (
        r"(kera)",
        "Kera",
        [],
    ),
    (
        r"(hoa\s*sen\s*home)",
        "HOA SEN HOME",
        [],
    ),

    # ============================================================
    # 7) LAST RESORT GENERIC RULES
    # ============================================================

    # Generic brand-ish terms, keep at bottom
    (
        r"\b(highlands|highland)\b",
        "HIGHLANDS COFFEE",
        [],
    ),
    (
        r"\b(pat[eê]|pate|pa\s*te)\b",
        "Pate",
        [],
    ),
]


def is_latin_letter(ch: str) -> bool:
    return (
        unicodedata.category(ch).startswith("L")
        and "LATIN" in unicodedata.name(ch, "")
    )

def clean_for_rule(text: str) -> str:
    keep_punct = set("&/+.-_%:")
    out = []

    for ch in str(text or ""):
        if is_latin_letter(ch) or ch.isdigit() or ch in keep_punct:
            out.append(ch)
        else:
            out.append(" ")

    text = "".join(out)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def extract_product_by_rules(ocr_text: str) -> str:
    text = clean_for_rule(ocr_text)

    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, text, flags=re.IGNORECASE):
            for line in lines:
                if re.search(re.escape(line.lower()), text, flags=re.IGNORECASE):
                    return f"{brand} {line}".strip()
            return brand

    return ""

tests = [
    ("HALONG CANFUCO j TIkTok @hanoinewsIheannzb HÀ NỘl NEWS", "Ha Long Canfoco Pate Cột Đèn"),

]
all_pass = True
for text, expected in tests:
    got = extract_product_by_rules(text)
    ok = got == expected
    all_pass = all_pass and ok
    status = "PASS" if ok else "FAIL"
    print(f"{status}: '{text[:45]}' -> got='{got}' | expected='{expected}'")

print()
print("All self-tests passed." if all_pass else "Some self-tests failed — check BRAND_RULES.")
print(f"Total rules loaded: {len(BRAND_RULES)}")


# Backward-compatible alias used by later cells
extract_product = extract_product_by_rules




FAIL: 'HALONG CANFUCO j TIkTok @hanoinewsIheannzb HÀ' -> got='' | expected='Ha Long Canfoco Pate Cột Đèn'

Some self-tests failed — check BRAND_RULES.
Total rules loaded: 108


## 5) Train lightweight product predictor from train labels



In [7]:
import re
import time
import unicodedata
from difflib import SequenceMatcher
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# ============================================================
# TEXT NORMALIZATION FOR MATCHING
# ============================================================

def normalize_match_text(text: str) -> str:
    """
    Normalize để fuzzy:
    - lowercase
    - bỏ dấu tiếng Việt
    - đ -> d
    - chỉ giữ a-z, 0-9, space
    """
    text = "" if text is None else str(text)
    text = text.lower()
    text = text.replace("đ", "d").replace("Đ", "D")
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_f1(a: str, b: str) -> float:
    a_tokens = set(normalize_match_text(a).split())
    b_tokens = set(normalize_match_text(b).split())

    if not a_tokens or not b_tokens:
        return 0.0

    common = a_tokens & b_tokens
    if not common:
        return 0.0

    precision = len(common) / len(b_tokens)
    recall = len(common) / len(a_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def partial_char_ratio(needle: str, haystack: str) -> float:
    """
    Pure-python partial ratio, nhẹ hơn rapidfuzz.
    So khớp product label với từng cửa sổ trong OCR.
    """
    needle = normalize_match_text(needle)
    haystack = normalize_match_text(haystack)

    if not needle or not haystack:
        return 0.0

    if needle in haystack:
        return 1.0

    if len(needle) > len(haystack):
        return SequenceMatcher(None, needle, haystack).ratio()

    n = len(needle)
    h = len(haystack)

    step = max(1, n // 4)
    best = 0.0

    for w in (n, int(n * 1.25), int(n * 1.5)):
        w = max(n, min(w, h))

        for start in range(0, h - w + 1, step):
            chunk = haystack[start:start + w]
            score = SequenceMatcher(None, needle, chunk).ratio()

            if score > best:
                best = score

            if best >= 0.98:
                return best

    return best


# ============================================================
# FUZZY PHRASE CORRECTOR
# ============================================================

class FuzzyPhraseCorrector:
    """
    Sửa cụm OCR bị méo về phrase canonical trước khi chạy rule/classifier.

    Ví dụ:
        OPTlPRO PLUS  -> optipro plus / opti pro plus
        CANF0CO       -> canfoco
        HIGHLAN       -> highlands

    Không dùng corrected_text để nộp OCR.
    Chỉ dùng corrected_text để predict product_name.
    """

    GENERIC_SKIP = {
        "pate",
        "cp",
        "nan",
        "milk",
        "sua",
        "coffee",
        "cafe",
        "gold",
        "plus",
        "pro",
        "new",
        "review",
        "official",
        "chinh",
        "hang",
        "mua",
        "ban",
        "gia",
        "sale",
    }

    MANUAL_PHRASES = [
        # Ha Long / Canfoco / Pate
        "ha long",
        "halong",
        "canfoco",
        "canfoco pate",
        "ha long canfoco",
        "ha long canfoco pate",
        "ha long canfoco pate cot den",
        "do hop ha long",
        "cong ty do hop ha long",
        "pate cot den",
        "pate cot den hai phong",
        "pate gan",
        "thit hop",
        "xuc xich",
        "lap xuong",
        "quang hong sardine",
        "sardine",

        # Nestle / NAN / milk powder
        "nestle",
        "nestle nan",
        "nan optipro",
        "nan optipro plus",
        "nan opti pro",
        "nan opti pro plus",
        "optipro",
        "optipro plus",
        "opti pro",
        "opti pro plus",
        "infinipro",
        "supreme pro",
        "alfamino",
        "aptamil",
        "pediasure",
        "ensure",
        "similac",
        "glucerna",
        "friso",
        "meiji",

        # Vietnamese milk brands
        "vinamilk",
        "vinamilk flex",
        "vinamilk adm gold",
        "vinamilk sure",
        "vinamilk canxi",
        "vinamilk dielac",
        "vinamilk grow",
        "vinamilk colosbaby",
        "ong tho",
        "th true milk",
        "dutch lady",
        "co gai ha lan",
        "nutifood",
        "grow plus",
        "ba vi",
        "lothamilk",
        "yomost",
        "dalat milk",
        "kun milk",
        "fami",
        "anlene",
        "anchor",

        # Highlands / coffee
        "highlands",
        "highlands coffee",
        "highland coffee",
        "tra sen vang",
        "tra vai",
        "tra sen vang tra vai",
        "banh mi highlands",
        "tra phuc kien",
        "americano vai",
        "the coffee house",
        "phuc long",
        "nescafe",
        "coffee mate",

        # Other observed
        "vissan",
        "acnes",
        "acnes vitamin cleanser",
        "hoa sen home",
        "hanayuki",
        "mailisa",
        "kera",
    ]

    def __init__(
        self,
        min_ngram=1,
        max_ngram=5,
        threshold=0.88,
        strong_threshold=0.94,
        margin=0.035,
    ):
        self.min_ngram = min_ngram
        self.max_ngram = max_ngram
        self.threshold = threshold
        self.strong_threshold = strong_threshold
        self.margin = margin

        self.phrase_counter = Counter()
        self.phrases = []
        self.len_index = defaultdict(list)

    def _add_phrase(self, phrase: str, weight: int = 1):
        norm = normalize_match_text(phrase)

        if not norm:
            return

        tokens = norm.split()
        if not tokens:
            return

        if len(tokens) == 1 and tokens[0] in self.GENERIC_SKIP:
            return

        if len(norm) <= 3:
            return

        self.phrase_counter[norm] += weight

    def fit(self, product_names):
        product_names = [
            "" if x is None else str(x).strip()
            for x in product_names
        ]
        product_names = [x for x in product_names if x]

        counts = Counter(product_names)

        # 1) Manual important phrases
        for phrase in self.MANUAL_PHRASES:
            self._add_phrase(phrase, weight=20)

        # 2) Phrases from product labels
        for product_name, cnt in counts.items():
            norm = normalize_match_text(product_name)
            tokens = norm.split()

            if not tokens:
                continue

            # Full product name
            self._add_phrase(norm, weight=cnt + 10)

            # N-grams inside product name
            max_n = min(self.max_ngram, len(tokens))

            for n in range(self.min_ngram, max_n + 1):
                for i in range(0, len(tokens) - n + 1):
                    phrase = " ".join(tokens[i:i + n])
                    self._add_phrase(phrase, weight=cnt)

        # 3) Build searchable index
        self.phrases = []
        self.len_index = defaultdict(list)

        for phrase, weight in self.phrase_counter.items():
            item = {
                "phrase": phrase,
                "tokens": set(phrase.split()),
                "len": len(phrase),
                "weight": weight,
            }

            self.phrases.append(item)
            self.len_index[len(phrase)].append(item)

        self.phrases.sort(
            key=lambda x: (len(x["tokens"]), x["len"], x["weight"]),
            reverse=True,
        )

        return self

    def _ratio(self, a: str, b: str) -> float:
        if not a or not b:
            return 0.0
        return SequenceMatcher(None, a, b).ratio()

    def _candidate_items_by_length(self, span: str):
        L = len(span)
        lo = max(1, int(L * 0.65))
        hi = int(L * 1.45) + 1

        candidates = []

        for length in range(lo, hi + 1):
            candidates.extend(self.len_index.get(length, []))

        return candidates

    def best_match(self, span: str):
        span = normalize_match_text(span)

        if not span:
            return "", 0.0, 0.0

        span_tokens = set(span.split())

        if not span_tokens:
            return "", 0.0, 0.0

        best_phrase = ""
        best_score = 0.0
        second_score = 0.0

        candidates = self._candidate_items_by_length(span)

        for item in candidates:
            phrase = item["phrase"]

            if span == phrase:
                return phrase, 1.0, 0.0

            phrase_tokens = item["tokens"]
            common = span_tokens & phrase_tokens

            char_score = self._ratio(span, phrase)

            if common:
                token_recall = len(common) / max(len(phrase_tokens), 1)
                token_precision = len(common) / max(len(span_tokens), 1)

                if token_recall + token_precision > 0:
                    token_score = 2 * token_recall * token_precision / (token_recall + token_precision)
                else:
                    token_score = 0.0

                score = max(
                    char_score,
                    0.70 * char_score + 0.30 * token_score,
                )
            else:
                # Không có token chung thì chỉ cho qua nếu char rất giống
                if char_score >= self.strong_threshold:
                    score = char_score
                else:
                    score = char_score * 0.75

            # Weight nhẹ theo tần suất phrase
            if item["weight"] >= 5:
                score += 0.005
            if item["weight"] >= 20:
                score += 0.005

            score = min(score, 1.0)

            if score > best_score:
                second_score = best_score
                best_score = score
                best_phrase = phrase
            elif score > second_score:
                second_score = score

        return best_phrase, best_score, second_score

    def correct(self, text: str, debug=False):
        norm = normalize_match_text(text)
        tokens = norm.split()

        if not tokens:
            return ("", []) if debug else ""

        out = []
        changes = []

        i = 0

        while i < len(tokens):
            best = None
            max_n = min(self.max_ngram, len(tokens) - i)

            # Check span dài trước
            for n in range(max_n, self.min_ngram - 1, -1):
                span_tokens = tokens[i:i + n]
                span = " ".join(span_tokens)

                if len(span) <= 3:
                    continue

                if n == 1 and span in self.GENERIC_SKIP:
                    continue

                phrase, score, second = self.best_match(span)

                if not phrase:
                    continue

                margin_ok = (score - second) >= self.margin
                strong_ok = score >= self.strong_threshold

                if score >= self.threshold and (margin_ok or strong_ok):
                    best = {
                        "from": span,
                        "to": phrase,
                        "score": round(float(score), 4),
                        "second": round(float(second), 4),
                        "n": n,
                    }
                    break

            if best is not None:
                out.extend(best["to"].split())

                if best["from"] != best["to"]:
                    changes.append(best)

                i += best["n"]
            else:
                out.append(tokens[i])
                i += 1

        corrected = " ".join(out)
        corrected = re.sub(r"\s+", " ", corrected).strip()

        if debug:
            return corrected, changes

        return corrected


# ============================================================
# FUZZY PRODUCT MATCHER
# ============================================================

class FuzzyProductMatcher:
    """
    Fuzzy match OCR text against canonical product names from train_labels.
    Trả về product_name cuối cùng, không phải phrase.
    """

    GENERIC_BLOCKLIST = {
        "pate",
        "cp",
        "nan",
        "milk",
        "sua",
        "coffee",
        "cafe",
        "nestle",
        "milo",
    }

    def __init__(
        self,
        min_label_count=1,
        min_norm_len=5,
        min_token_len=2,
    ):
        self.min_label_count = min_label_count
        self.min_norm_len = min_norm_len
        self.min_token_len = min_token_len
        self.items = []

    def fit(self, product_names):
        product_names = [
            "" if x is None else str(x).strip()
            for x in product_names
        ]
        product_names = [x for x in product_names if x]

        counts = Counter(product_names)
        canonical_by_norm = {}

        for name, cnt in counts.items():
            if cnt < self.min_label_count:
                continue

            norm = normalize_match_text(name)

            if not norm:
                continue

            tokens = norm.split()

            if norm in self.GENERIC_BLOCKLIST:
                continue

            if len(norm) < self.min_norm_len and len(tokens) < self.min_token_len:
                continue

            if norm not in canonical_by_norm:
                canonical_by_norm[norm] = (name, cnt)
            else:
                old_name, old_cnt = canonical_by_norm[norm]
                if cnt > old_cnt:
                    canonical_by_norm[norm] = (name, cnt)

        self.items = []

        for norm, (name, cnt) in canonical_by_norm.items():
            self.items.append({
                "name": name,
                "norm": norm,
                "tokens": set(norm.split()),
                "count": cnt,
            })

        self.items.sort(
            key=lambda x: (len(x["tokens"]), len(x["norm"]), x["count"]),
            reverse=True,
        )

        return self

    def score_one(self, ocr_text_norm: str, item: dict) -> float:
        label_norm = item["norm"]

        if not label_norm or not ocr_text_norm:
            return 0.0

        if label_norm in ocr_text_norm:
            return 1.0

        label_tokens = item["tokens"]
        ocr_tokens = set(ocr_text_norm.split())

        if not label_tokens:
            return 0.0

        common = label_tokens & ocr_tokens

        if len(label_tokens) >= 2 and label_tokens.issubset(ocr_tokens):
            return 0.97

        recall = len(common) / len(label_tokens)
        precision = len(common) / max(len(ocr_tokens), 1)

        tok_score = 0.0
        if recall + precision > 0:
            tok_score = 2 * recall * precision / (recall + precision)

        char_score = partial_char_ratio(label_norm, ocr_text_norm)

        score = max(
            char_score,
            0.65 * char_score + 0.35 * recall,
            tok_score,
        )

        if len(common) == 0 and char_score < 0.92:
            score *= 0.75

        return float(score)

    def match(self, ocr_text: str):
        ocr_text_norm = normalize_match_text(ocr_text)

        if not ocr_text_norm or not self.items:
            return "", 0.0

        best_name = ""
        best_score = 0.0

        for item in self.items:
            score = self.score_one(ocr_text_norm, item)

            if score > best_score:
                best_score = score
                best_name = item["name"]

            if best_score >= 0.995:
                break

        return best_name, best_score


# ============================================================
# PRODUCT PREDICTOR WITH PHRASE-CORRECT + RULE + FUZZY + ML
# ============================================================

class ProductPredictor:
    def __init__(
        self,
        min_class_count=3,
        prob_threshold=0.60,
        max_features=3000,
        fuzzy_threshold=0.88,
        fuzzy_strong_threshold=0.94,
        fuzzy_min_label_count=1,
        phrase_threshold=0.88,
        phrase_strong_threshold=0.94,
        phrase_margin=0.035,
    ):
        self.min_class_count = min_class_count
        self.prob_threshold = prob_threshold
        self.max_features = max_features

        self.fuzzy_threshold = fuzzy_threshold
        self.fuzzy_strong_threshold = fuzzy_strong_threshold
        self.fuzzy_min_label_count = fuzzy_min_label_count

        self.phrase_threshold = phrase_threshold
        self.phrase_strong_threshold = phrase_strong_threshold
        self.phrase_margin = phrase_margin

        self._has_clf = None
        self._prod_clf = None
        self._fuzzy = None
        self._phrase_corrector = None
        self._rule_fn = None

        self._n_train = 0
        self._n_classes = 0
        self._n_fuzzy_items = 0
        self._n_phrase_items = 0

    def fit(self, train_labels, rule_fn):
        df = train_labels.copy()

        df["ocr_text"] = df["ocr_text"].fillna("").astype(str).str.strip()
        df["product_name"] = df["product_name"].fillna("").astype(str).str.strip()

        self._rule_fn = rule_fn

        # 1) Binary classifier: OCR này có product hay không?
        self._has_clf = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(2, 4),
                    max_features=self.max_features,
                    min_df=2,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=400,
                    class_weight="balanced",
                ),
            ),
        ])

        y_has = (df["product_name"] != "").astype(int)

        if y_has.nunique() >= 2:
            self._has_clf.fit(df["ocr_text"], y_has)
        else:
            self._has_clf = None

        # 2) Positive rows
        pos = df[
            (df["ocr_text"] != "")
            & (df["product_name"] != "")
        ].copy()

        # 3) Phrase corrector: sửa cụm OCR sai trước rule/classifier
        self._phrase_corrector = FuzzyPhraseCorrector(
            min_ngram=1,
            max_ngram=5,
            threshold=self.phrase_threshold,
            strong_threshold=self.phrase_strong_threshold,
            margin=self.phrase_margin,
        )
        self._phrase_corrector.fit(pos["product_name"].tolist())
        self._n_phrase_items = len(self._phrase_corrector.phrases)

        # 4) Product-level fuzzy matcher
        self._fuzzy = FuzzyProductMatcher(
            min_label_count=self.fuzzy_min_label_count,
        )
        self._fuzzy.fit(pos["product_name"].tolist())
        self._n_fuzzy_items = len(self._fuzzy.items)

        # 5) Multi-class product classifier
        keep = pos["product_name"].value_counts()
        keep = keep[keep >= self.min_class_count].index
        pos_cls = pos[pos["product_name"].isin(keep)].copy()

        self._prod_clf = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(2, 4),
                    max_features=self.max_features,
                    min_df=2,
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=400,
                    class_weight="balanced",
                ),
            ),
        ])

        if len(pos_cls) and pos_cls["product_name"].nunique() >= 2:
            self._prod_clf.fit(pos_cls["ocr_text"], pos_cls["product_name"])
            self._n_classes = pos_cls["product_name"].nunique()
        else:
            self._prod_clf = None
            self._n_classes = 0

        self._n_train = len(df)

        return self

    def _correct_text(self, ocr_text: str):
        if self._phrase_corrector is None:
            return ocr_text, []

        corrected_text, changes = self._phrase_corrector.correct(ocr_text, debug=True)
        return corrected_text, changes

    def _has_product_prob(self, ocr_text: str) -> float:
        if self._has_clf is None:
            return 1.0

        proba = self._has_clf.predict_proba([ocr_text])[0]
        classes = list(self._has_clf.classes_)

        if 1 not in classes:
            return 0.0

        return float(proba[classes.index(1)])

    def predict(self, ocr_text):
        ocr_text = "" if ocr_text is None else str(ocr_text).strip()

        if not ocr_text:
            return ""

        corrected_text, _changes = self._correct_text(ocr_text)

        # 1) Rule first, run on corrected text first
        ruled = self._rule_fn(corrected_text) if self._rule_fn is not None else ""

        # fallback to raw if corrected text did not match
        if not ruled and corrected_text != ocr_text:
            ruled = self._rule_fn(ocr_text) if self._rule_fn is not None else ""

        if ruled:
            return ruled

        # 2) Product-level fuzzy strong
        fuzzy_pred = ""
        fuzzy_score = 0.0

        if self._fuzzy is not None:
            fuzzy_pred, fuzzy_score = self._fuzzy.match(corrected_text)

            if fuzzy_pred and fuzzy_score >= self.fuzzy_strong_threshold:
                return fuzzy_pred

        # 3) Has-product gate
        if self._has_clf is not None:
            has_prob_raw = self._has_product_prob(ocr_text)
            has_prob_fixed = self._has_product_prob(corrected_text)
            has_prob = max(has_prob_raw, has_prob_fixed)

            if has_prob < self.prob_threshold:
                return ""

        # 4) Product-level fuzzy normal
        if fuzzy_pred and fuzzy_score >= self.fuzzy_threshold:
            return fuzzy_pred

        # 5) Product classifier fallback
        if self._prod_clf is not None:
            return str(self._prod_clf.predict([corrected_text])[0])

        return ""

    def predict_debug(self, ocr_text):
        """
        Debug toàn bộ route:
        - raw OCR
        - corrected text
        - phrase changes
        - rule
        - fuzzy product
        - has-product prob
        - final prediction
        """
        ocr_text = "" if ocr_text is None else str(ocr_text).strip()

        info = {
            "ocr_text": ocr_text,
            "corrected_text": "",
            "phrase_changes": [],
            "rule_pred_corrected": "",
            "rule_pred_raw": "",
            "fuzzy_pred": "",
            "fuzzy_score": 0.0,
            "has_product_prob_raw": 0.0,
            "has_product_prob_fixed": 0.0,
            "has_product_prob": 0.0,
            "final_pred": "",
            "route": "",
        }

        if not ocr_text:
            info["route"] = "empty_ocr"
            return info

        corrected_text, changes = self._correct_text(ocr_text)
        info["corrected_text"] = corrected_text
        info["phrase_changes"] = changes

        # Rule on corrected
        if self._rule_fn is not None:
            info["rule_pred_corrected"] = self._rule_fn(corrected_text)
            info["rule_pred_raw"] = self._rule_fn(ocr_text)

        if info["rule_pred_corrected"]:
            info["final_pred"] = info["rule_pred_corrected"]
            info["route"] = "rule_corrected"
            return info

        if info["rule_pred_raw"]:
            info["final_pred"] = info["rule_pred_raw"]
            info["route"] = "rule_raw"
            return info

        # Product fuzzy
        if self._fuzzy is not None:
            fuzzy_pred, fuzzy_score = self._fuzzy.match(corrected_text)
            info["fuzzy_pred"] = fuzzy_pred
            info["fuzzy_score"] = round(float(fuzzy_score), 4)

            if fuzzy_pred and fuzzy_score >= self.fuzzy_strong_threshold:
                info["final_pred"] = fuzzy_pred
                info["route"] = "fuzzy_strong"
                return info

        # Has-product probability
        if self._has_clf is not None:
            has_prob_raw = self._has_product_prob(ocr_text)
            has_prob_fixed = self._has_product_prob(corrected_text)
            has_prob = max(has_prob_raw, has_prob_fixed)

            info["has_product_prob_raw"] = round(float(has_prob_raw), 4)
            info["has_product_prob_fixed"] = round(float(has_prob_fixed), 4)
            info["has_product_prob"] = round(float(has_prob), 4)

            if has_prob < self.prob_threshold:
                info["route"] = "no_product_gate"
                return info
        else:
            info["has_product_prob_raw"] = 1.0
            info["has_product_prob_fixed"] = 1.0
            info["has_product_prob"] = 1.0

        if info["fuzzy_pred"] and info["fuzzy_score"] >= self.fuzzy_threshold:
            info["final_pred"] = info["fuzzy_pred"]
            info["route"] = "fuzzy_after_gate"
            return info

        if self._prod_clf is not None:
            clf_pred = str(self._prod_clf.predict([corrected_text])[0])
            info["final_pred"] = clf_pred
            info["route"] = "product_clf"
            return info

        info["route"] = "no_prediction"
        return info

    def debug_correction(self, ocr_text: str):
        corrected_text, changes = self._correct_text(ocr_text)

        return {
            "ocr_text": "" if ocr_text is None else str(ocr_text).strip(),
            "corrected_text": corrected_text,
            "changes": changes,
            "final_pred": self.predict(ocr_text),
        }

    def summary(self):
        return {
            "n_train": self._n_train,
            "n_classes": self._n_classes,
            "n_fuzzy_items": self._n_fuzzy_items,
            "n_phrase_items": self._n_phrase_items,
            "min_class_count": self.min_class_count,
            "prob_threshold": self.prob_threshold,
            "fuzzy_threshold": self.fuzzy_threshold,
            "fuzzy_strong_threshold": self.fuzzy_strong_threshold,
            "phrase_threshold": self.phrase_threshold,
            "phrase_strong_threshold": self.phrase_strong_threshold,
            "phrase_margin": self.phrase_margin,
            "max_features": self.max_features,
        }


# ============================================================
# TRAIN / INIT PRODUCT PREDICTOR
# ============================================================

def _get_rule_fn():
    """
    Hỗ trợ cả 2 tên hàm:
    - extract_product_by_rules
    - extract_product
    """
    if "extract_product_by_rules" in globals():
        return extract_product_by_rules



    raise NameError("No rule function found. Please define extract_product_by_rules or extract_product first.")


RULE_FN = _get_rule_fn()

product_predictor = None
_rules_only_corrector = FuzzyPhraseCorrector(
    min_ngram=1,
    max_ngram=5,
    threshold=0.88,
    strong_threshold=0.94,
    margin=0.035,
)
_rules_only_corrector.fit([])

_train_labels_available = (
    "train_labels_df" in globals()
    and train_labels_df is not None
)

if _train_labels_available:
    product_predictor = ProductPredictor(
        min_class_count=3,
        prob_threshold=0.60,
        max_features=3000,
        fuzzy_threshold=0.88,
        fuzzy_strong_threshold=0.94,
        fuzzy_min_label_count=1,
        phrase_threshold=0.88,
        phrase_strong_threshold=0.94,
        phrase_margin=0.035,
    )

    product_predictor.fit(train_labels_df, RULE_FN)

    pos = train_labels_df.copy()
    pos["ocr_text"] = pos["ocr_text"].fillna("").astype(str).str.strip()
    pos["product_name"] = pos["product_name"].fillna("").astype(str).str.strip()
    n_pairs = ((pos["ocr_text"] != "") & (pos["product_name"] != "")).sum()

    print(f"Trained lightweight product head on {len(train_labels_df):,} rows ({n_pairs:,} OCR+product pairs)")
    print(product_predictor.summary())

    # Small timing check
    t0 = time.perf_counter()
    for _ in range(200):
        product_predictor.predict("Vinamilk Flex 180ml Ha Long Canfoco NESTLE NAN OPTlPRO PLUS")
    ms = (time.perf_counter() - t0) / 200 * 1000
    print(f"Product inference: ~{ms:.2f} ms/image CPU after OCR")

else:
    print("train_labels.csv not found — rules-only mode with manual phrase corrector")


def predict_product(ocr_text: str) -> str:
    if product_predictor is not None:
        return product_predictor.predict(ocr_text)

    raw = "" if ocr_text is None else str(ocr_text).strip()

    if not raw:
        return ""

    corrected = _rules_only_corrector.correct(raw)

    ruled = RULE_FN(corrected)
    if ruled:
        return ruled

    return RULE_FN(raw)



print("Product predictor ready.")




Trained lightweight product head on 4,892 rows (2,876 OCR+product pairs)
{'n_train': 4892, 'n_classes': 249, 'n_fuzzy_items': 327, 'n_phrase_items': 2205, 'min_class_count': 3, 'prob_threshold': 0.6, 'fuzzy_threshold': 0.88, 'fuzzy_strong_threshold': 0.94, 'phrase_threshold': 0.88, 'phrase_strong_threshold': 0.94, 'phrase_margin': 0.035, 'max_features': 3000}
Product inference: ~343.14 ms/image CPU after OCR
Product predictor ready.


## 5.5) Final product head: rule/fuzzy only, no Logistic Regression fallback

Cell này là khác biệt chính của bản final. Nó giữ product post-processing bằng rule/fuzzy, nhưng không cho `LogisticRegression.predict()` tự chọn nhãn khi không chắc.




In [ ]:
# ============================================================
# FINAL PRODUCT POLICY
# ============================================================
# Empirical result from our submissions:
#   - PaddleOCR GPU + LR fallback     : ~0.6717
#   - PaddleOCR GPU + no-LR fallback  : ~0.6732
# Therefore final pipeline disables LogisticRegression fallback.
# OCR text is NOT changed here. Only product_name is affected.

PRODUCT_MODE = "rule_fuzzy_no_lr_badlabel_guard"

# Labels clearly outside the official Pate/Milk domain or obvious TikTok/news/music noise.
# Keep this list conservative. Do not use a strict whitelist, because strict whitelist blanked too much.
BAD_PRODUCT_PATTERNS = [
    "chinsu", "tương ớt", "tuong ot",
    "weetes", "index", "vietnamindex",
    "imcv", "sconnect", "music", "bebop", "breaking news",
    "mailisa", "hanayuki", "ozzies",
    "thuốc", "thuoc", "y học cổ truyền", "y hoc co truyen",
    "kiến thức kinh tế", "kien thuc kinh te","coffee"
]

SAFE_PRODUCT_ROUTES = {
    "rule_corrected",
    "rule_raw",
    "fuzzy_strong",
    "fuzzy_after_gate",
}


def _norm_guard_text(x: str) -> str:
    x = "" if x is None else str(x).lower()
    x = x.replace("đ", "d")
    x = unicodedata.normalize("NFD", x)
    x = "".join(ch for ch in x if unicodedata.category(ch) != "Mn")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def is_bad_product_name(product: str) -> bool:
    p = _norm_guard_text(product)
    if not p:
        return False
    return any(_norm_guard_text(bad) in p for bad in BAD_PRODUCT_PATTERNS)


def predict_product_stable(ocr_text: str) -> str:
    """
    Final product extractor:
    - Use existing ProductPredictor debug route.
    - Accept only rule/fuzzy routes.
    - Reject LogisticRegression route = product_clf.
    - Reject obvious out-of-domain/noise labels.
    - Do not modify OCR text.
    """
    raw = "" if ocr_text is None else str(ocr_text).strip()
    if not raw:
        return ""

    if product_predictor is not None:
        info = product_predictor.predict_debug(raw)
        route = info.get("route", "")
        pred = "" if info.get("final_pred") is None else str(info.get("final_pred")).strip()

        if route in SAFE_PRODUCT_ROUTES and pred and not is_bad_product_name(pred):
            return pred

        # Explicitly block LogisticRegression fallback and unclear/noise predictions.
        return ""

    # If train_labels.csv is unavailable, stay rules-only.
    corrected = _rules_only_corrector.correct(raw)
    ruled = RULE_FN(corrected) or RULE_FN(raw)
    if ruled and not is_bad_product_name(ruled):
        return ruled
    return ""


# Override global function used by run_ocr().
predict_product = predict_product_stable

print("Final product policy enabled:", PRODUCT_MODE)
print("Accepted routes:", sorted(SAFE_PRODUCT_ROUTES))
print("Blocked route: product_clf / LogisticRegression fallback")




Final product policy enabled: rule_fuzzy_no_lr_badlabel_guard
Accepted routes: ['fuzzy_after_gate', 'fuzzy_strong', 'rule_corrected', 'rule_raw']
Blocked route: product_clf / LogisticRegression fallback


## 6) PaddleOCR GPU engine



In [15]:
import os, re, time, tempfile
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

import paddle

# Reuse safe PaddleOCR import from version-check cell. If this cell is run alone, define a local fallback.
if "PaddleOCR" not in globals():
    try:
        paddleocr, PaddleOCR = safe_import_paddleocr(hide_torch=True)
    except NameError:
        import sys, importlib.util as _importlib_util
        original_find_spec = _importlib_util.find_spec
        def patched_find_spec(name, *args, **kwargs):
            if name == "torch" or name.startswith("torch."):
                return None
            return original_find_spec(name, *args, **kwargs)
        _importlib_util.find_spec = patched_find_spec
        try:
            import paddleocr
            from paddleocr import PaddleOCR
        finally:
            _importlib_util.find_spec = original_find_spec

# ------------------------------------------------------------
# OCR config
# ------------------------------------------------------------
OCR_SCORE_THRESHOLD = 0.30   # lower = more text, higher = cleaner but may miss noisy text
MAX_DIM = 1280              # resize longest side before OCR; 1280 is a good speed/accuracy tradeoff
USE_SHARPEN = True
USE_CONTRAST = True

if paddle.device.is_compiled_with_cuda():
    try:
        paddle.set_device("gpu:0")
        OCR_DEVICE = "gpu"
    except Exception as e:
        print("Cannot set gpu:0, fallback CPU:", repr(e))
        OCR_DEVICE = "cpu"
else:
    OCR_DEVICE = "cpu"

print("OCR_DEVICE:", OCR_DEVICE)


def make_paddle_ocr(device: str = OCR_DEVICE):
    """Create PaddleOCR object with a few config fallbacks for PaddleOCR 3.x / older signatures."""
    configs = [
        dict(
            lang="vi",
            device=device,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            engine="paddle",
        ),
        dict(
            lang="vi",
            device=device,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        ),
        dict(
            lang="vi",
            use_angle_cls=False,
            show_log=False,
            use_gpu=(device == "gpu"),
        ),
    ]

    last_error = None
    for cfg in configs:
        try:
            print("Trying PaddleOCR config:", cfg)
            reader = PaddleOCR(**cfg)
            print("PaddleOCR loaded OK")
            return reader
        except TypeError as e:
            last_error = e
            print("Config rejected:", repr(e))
        except Exception as e:
            last_error = e
            print("Config failed:", repr(e))
    raise RuntimeError(f"Cannot initialize PaddleOCR. Last error: {last_error!r}")


paddle_reader = make_paddle_ocr(OCR_DEVICE)


def image_path_for_id(image_id: str) -> Path:
    # Most files are .jpg. Keep fallback for jpeg/png if local copy differs.
    for ext in (".jpg", ".jpeg", ".png", ".webp"):
        p = IMAGES_DIR / f"{image_id}{ext}"
        if p.exists():
            return p
    return IMAGES_DIR / f"{image_id}.jpg"


def load_image(image_id: str):
    path = image_path_for_id(image_id)
    if not path.exists():
        return None
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        print(f"Image load failed for {image_id}: {repr(e)}")
        return None


def preprocess(img: Image.Image, max_dim: int = MAX_DIM):
    """
    OCR-safe preprocessing:
    - Upscale ảnh nhỏ để chữ to hơn.
    - CLAHE nhẹ trên kênh sáng để tăng contrast chữ.
    - Sharpen nhẹ để rõ viền.
    - Không canonical text, không threshold gắt.
    """
    import cv2
    import numpy as np
    from PIL import ImageEnhance, ImageFilter

    img = img.convert("RGB")
    w, h = img.size

    # 1) Resize / zoom
    # Nếu ảnh nhỏ hoặc text thumbnail nhỏ, upscale giúp PaddleOCR đọc tốt hơn.
    longest = max(w, h)

    if longest < 960:
        scale = min(2.0, 960 / max(1, longest))
        new_w = max(1, int(w * scale))
        new_h = max(1, int(h * scale))
        img = img.resize((new_w, new_h), Image.BICUBIC)
        w, h = img.size

    # Sau khi upscale, vẫn cap max_dim để không quá nặng.
    longest = max(w, h)
    if longest > max_dim:
        ratio = max_dim / longest
        img = img.resize(
            (max(1, int(w * ratio)), max(1, int(h * ratio))),
            Image.BICUBIC,
        )

    # 2) CLAHE nhẹ trên luminance channel
    # Tốt hơn tăng contrast toàn ảnh vì không phá màu quá mạnh.
    arr = np.array(img)
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=1.8,
        tileGridSize=(8, 8),
    )
    l2 = clahe.apply(l)

    lab2 = cv2.merge([l2, a, b])
    arr2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
    img = Image.fromarray(arr2)

    # 3) Contrast nhẹ
    img = ImageEnhance.Contrast(img).enhance(1.15)

    # 4) Sharpen nhẹ, tránh quá tay gây halo/nhiễu
    img = img.filter(ImageFilter.UnsharpMask(
        radius=1.0,
        percent=120,
        threshold=3,
    ))

    return img


def postprocess_ocr(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return ""
    # Remove consecutive duplicate tokens, common in thumbnails.
    tokens = text.split()
    out = []
    for tok in tokens:
        if not out or tok.lower() != out[-1].lower():
            out.append(tok)
    return " ".join(out)


def _extract_from_v3_dict(data: dict, score_threshold: float) -> list[str]:
    data = data.get("res", data)
    texts = data.get("rec_texts") or data.get("texts") or []
    scores = data.get("rec_scores") or data.get("scores") or []
    if texts and not scores:
        scores = [1.0] * len(texts)
    lines = []
    for text, score in zip(texts, scores):
        text = str(text).strip()
        try:
            score = float(score)
        except Exception:
            score = 0.0
        if text and score >= score_threshold:
            lines.append(text)
    return lines


def _extract_from_legacy_list(obj, score_threshold: float) -> list[str]:
    """Parse PaddleOCR 2.x style: [[box, (text, score)], ...] or nested pages."""
    lines = []
    if obj is None:
        return lines
    if isinstance(obj, (tuple, list)):
        # Item: [box, (text, score)]
        if len(obj) >= 2 and isinstance(obj[1], (tuple, list)) and len(obj[1]) >= 2:
            text, score = obj[1][0], obj[1][1]
            try:
                score = float(score)
            except Exception:
                score = 0.0
            if str(text).strip() and score >= score_threshold:
                lines.append(str(text).strip())
        else:
            for x in obj:
                lines.extend(_extract_from_legacy_list(x, score_threshold))
    return lines


def parse_paddle_output(output, score_threshold: float = OCR_SCORE_THRESHOLD) -> str:
    if output is None:
        return ""

    lines = []
    outputs = output if isinstance(output, list) else [output]

    for res in outputs:
        data = None

        # PaddleOCR 3.x result object may expose .json dict/property or method.
        if hasattr(res, "json"):
            try:
                data = res.json() if callable(res.json) else res.json
            except Exception:
                data = None

        if data is None and hasattr(res, "to_dict"):
            try:
                data = res.to_dict()
            except Exception:
                data = None

        if data is None and isinstance(res, dict):
            data = res

        if isinstance(data, dict):
            lines.extend(_extract_from_v3_dict(data, score_threshold))
        else:
            lines.extend(_extract_from_legacy_list(res, score_threshold))

    return postprocess_ocr(" ".join(lines))


def run_ocr(image_id: str) -> tuple[str, str]:
    img = load_image(image_id)
    if img is None:
        return "", ""

    img = preprocess(img)
    img_np = np.array(img)

    try:
        # PaddleOCR 3.x preferred API.
        output = paddle_reader.predict(img_np)
    except AttributeError:
        # Older API fallback. Do not pass cls=..., because newer versions reject it.
        output = paddle_reader.ocr(img_np)
    except Exception as e:
        print(f"OCR failed for {image_id}: {repr(e)}")
        return "", ""

    ocr_text = parse_paddle_output(output, score_threshold=OCR_SCORE_THRESHOLD)
    product_name = predict_product(ocr_text)
    return ocr_text, product_name


# Smoke test on first available image.
first_id = test_df["image_id"].iloc[0]
ocr_text, product_name = run_ocr(first_id)
print("\nSmoke test")
print("image_id    :", first_id)
print("ocr_text    :", ocr_text[:300] + ("..." if len(ocr_text) > 300 else ""))
print("product_name:", product_name or "(empty)")



Creating model: ('PP-OCRv6_medium_det', None, 'paddle')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


OCR_DEVICE: gpu
Trying PaddleOCR config: {'lang': 'vi', 'device': 'gpu', 'use_doc_orientation_classify': False, 'use_doc_unwarping': False, 'use_textline_orientation': False, 'engine': 'paddle'}


Creating model: ('PP-OCRv6_medium_rec', None, 'paddle')
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


PaddleOCR loaded OK

Smoke test
image_id    : img_2934
ocr_text    : THEOTIENTHANG-VICNEWS ÀNH:TONG HOP HA LONG CANFOCO ISO 22000:2018 FSSC 22000 oate TikTok @hanoinews.theanh28 HÀ NÔI pate NEWS NGAY DANG:11/01/2026 CHÍNH THÚC: NHÀ MÁY DÕ HŹP H LONG TAM DÙNG HOAT DÔNG TÙ 12/1
product_name: Ha Long Canfoco


## 6.5) Hybrid OCR: Paddle base + VietOCR rescue

Cell này override `run_ocr()` theo hướng conservative: PaddleOCR vẫn là base; VietOCR chỉ được dùng khi Paddle blank/yếu và text VietOCR pass quality filter.


In [10]:
# ============================================================
# HYBRID OCR POLICY: PaddleOCR base + VietOCR rescue only
# ============================================================
# Important:
# - This cell does NOT change product policy. product_name still uses predict_product(),
#   which was overridden earlier to rule/fuzzy no-LR.
# - VietOCR has no confidence in this notebook; we use heuristic quality gates.
# - VietOCR runs on CPU-only torch by default, so we call it only when Paddle is weak.

USE_VIETOCR_RESCUE = True
VIETOCR_DEVICE = "cpu"
VIETOCR_MIN_QUALITY = 0.70
PADDLE_GOOD_MIN_AVG_SCORE = 0.38
PADDLE_MIN_CHARS_FOR_GOOD = 8
VIETOCR_MAX_CROPS = 20
VIETOCR_CROP_PAD_RATIO = 0.03

try:
    from vietocr.tool.config import Cfg
    from vietocr.tool.predictor import Predictor
    _vietocr_available = True
except Exception as e:
    print("[warn] VietOCR import failed; rescue disabled:", repr(e))
    _vietocr_available = False

vietocr_reader = None


def make_vietocr_predictor(device: str = VIETOCR_DEVICE):
    cfg = Cfg.load_config_from_name("vgg_transformer")
    cfg["device"] = device
    if "cnn" in cfg and isinstance(cfg["cnn"], dict):
        cfg["cnn"]["pretrained"] = False
    if "predictor" in cfg and isinstance(cfg["predictor"], dict):
        cfg["predictor"]["beamsearch"] = False
    return Predictor(cfg)


if USE_VIETOCR_RESCUE and _vietocr_available:
    try:
        vietocr_reader = make_vietocr_predictor(VIETOCR_DEVICE)
        print("VietOCR rescue loaded on", VIETOCR_DEVICE)
    except Exception as e:
        print("[warn] Cannot initialize VietOCR; rescue disabled:", repr(e))
        vietocr_reader = None


# -----------------------------
# Paddle detail parser
# -----------------------------

def _result_to_data(res):
    data = None
    if hasattr(res, "json"):
        try:
            data = res.json() if callable(res.json) else res.json
        except Exception:
            data = None
    if data is None and hasattr(res, "to_dict"):
        try:
            data = res.to_dict()
        except Exception:
            data = None
    if data is None and isinstance(res, dict):
        data = res
    if isinstance(data, dict) and "res" in data and isinstance(data["res"], dict):
        data = data["res"]
    return data


def _poly_to_np(poly):
    try:
        arr = np.array(poly, dtype=np.float32)
        if arr.ndim == 2 and arr.shape[0] >= 4 and arr.shape[1] >= 2:
            return arr[:, :2]
    except Exception:
        return None
    return None


def parse_paddle_detail(output, score_threshold: float = OCR_SCORE_THRESHOLD):
    """
    Return:
      text, avg_score, lines, scores, polys
    Tries to parse PaddleOCR 3.x dict result and legacy list output.
    """
    lines, scores, polys = [], [], []
    outputs = output if isinstance(output, list) else [output]

    def add_line(text, score, poly=None):
        text = str(text).strip()
        try:
            score = float(score)
        except Exception:
            score = 0.0
        if text and score >= score_threshold:
            lines.append(text)
            scores.append(score)
            if poly is not None:
                arr = _poly_to_np(poly)
                if arr is not None:
                    polys.append(arr)

    for res in outputs:
        data = _result_to_data(res)
        if isinstance(data, dict):
            texts = data.get("rec_texts") or data.get("texts") or []
            rec_scores = data.get("rec_scores") or data.get("scores") or []
            # Depending on PaddleOCR version, polygons may be in one of these fields.
            raw_polys = (
                data.get("rec_polys")
                or data.get("dt_polys")
                or data.get("rec_boxes")
                or data.get("dt_boxes")
                or []
            )
            if texts and not rec_scores:
                rec_scores = [1.0] * len(texts)
            for idx, text in enumerate(texts):
                sc = rec_scores[idx] if idx < len(rec_scores) else 1.0
                poly = raw_polys[idx] if idx < len(raw_polys) else None
                add_line(text, sc, poly)
        else:
            # Legacy format recursive parser.
            def rec(obj):
                if obj is None:
                    return
                if isinstance(obj, (tuple, list)):
                    if len(obj) >= 2 and isinstance(obj[1], (tuple, list)) and len(obj[1]) >= 2:
                        poly = obj[0]
                        text, sc = obj[1][0], obj[1][1]
                        add_line(text, sc, poly)
                    else:
                        for x in obj:
                            rec(x)
            rec(res)

    text = postprocess_ocr(" ".join(lines))
    avg = float(np.mean(scores)) if scores else 0.0
    return text, avg, lines, scores, polys


# -----------------------------
# Text quality gates
# -----------------------------

def text_quality_score(text: str) -> float:
    s = "" if text is None else str(text).strip()
    low = s.lower()
    if not s:
        return 0.0

    score = 0.0
    if len(s) >= 8:
        score += 0.25
    if len(s.split()) >= 2:
        score += 0.20

    domain_keywords = [
        "sữa", "sua", "milk", "nan", "nestle", "nestlé", "vinamilk",
        "aptamil", "ensure", "similac", "pediasure", "friso", "meiji",
        "pate", "patê", "pa te", "ha long", "hạ long", "canfoco",
        "cột đèn", "cot den", "vissan", "đồ hộp", "do hop", "thịt hộp", "thit hop",
    ]
    if any(k in low for k in domain_keywords):
        score += 0.35

    bad_terms = [
        "file", "stat", "start", "030000", "001900", "000099",
        "music", "breaking news", "index", "coffee", "highlands",
        "sconnect", "imcv", "bebop",
    ]
    if any(x in low for x in bad_terms):
        score -= 0.40

    alnum = sum(ch.isalnum() for ch in s)
    digit = sum(ch.isdigit() for ch in s)
    if alnum > 0 and digit / alnum > 0.65:
        score -= 0.30
    if len(s) <= 3:
        score -= 0.50

    return max(0.0, min(1.0, score))


def is_good_paddle_text(text: str, avg_score: float) -> bool:
    s = "" if text is None else str(text).strip()
    if not s:
        return False
    if avg_score >= PADDLE_GOOD_MIN_AVG_SCORE and len(s) >= PADDLE_MIN_CHARS_FOR_GOOD:
        return True
    # If it contains domain keywords, keep it even if noisy.
    return text_quality_score(s) >= 0.70


# -----------------------------
# Crop + VietOCR recognition
# -----------------------------

def crop_poly_bbox(img: Image.Image, poly, pad_ratio: float = VIETOCR_CROP_PAD_RATIO):
    arr = _poly_to_np(poly)
    if arr is None:
        return None
    w, h = img.size
    x1, y1 = float(arr[:, 0].min()), float(arr[:, 1].min())
    x2, y2 = float(arr[:, 0].max()), float(arr[:, 1].max())
    bw, bh = x2 - x1, y2 - y1
    pad = max(2, int(max(bw, bh) * pad_ratio))
    x1 = max(0, int(x1) - pad)
    y1 = max(0, int(y1) - pad)
    x2 = min(w, int(x2) + pad)
    y2 = min(h, int(y2) + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return img.crop((x1, y1, x2, y2))


def recognize_vietocr_on_polys(img: Image.Image, polys: list) -> str:
    if vietocr_reader is None or not polys:
        return ""

    # Limit crops to avoid CPU VietOCR being too slow.
    selected = polys[:VIETOCR_MAX_CROPS]
    lines = []
    for poly in selected:
        crop = crop_poly_bbox(img, poly)
        if crop is None:
            continue
        try:
            text = vietocr_reader.predict(crop)
        except Exception as e:
            print("[warn] VietOCR crop failed:", repr(e))
            continue
        text = re.sub(r"\s+", " ", str(text)).strip()
        if text_quality_score(text) >= 0.35:  # line-level loose; image-level gate is stricter.
            lines.append(text)
    return postprocess_ocr(" ".join(lines))


def merge_paddle_vietocr(paddle_text: str, paddle_avg: float, viet_text: str):
    """Return final OCR text and route."""
    if is_good_paddle_text(paddle_text, paddle_avg):
        return paddle_text, "paddle_good"

    q_viet = text_quality_score(viet_text)
    if q_viet >= VIETOCR_MIN_QUALITY:
        return viet_text, f"vietocr_rescue_q{q_viet:.2f}"

    return paddle_text, "paddle_fallback"


# -----------------------------
# Optional ResNet visual gate hook
# -----------------------------

def apply_resnet_gate(product_name: str, image_id: str, img: Image.Image) -> str:
    """No-op by default. Optional ResNet cell can override this function."""
    return product_name


# -----------------------------
# Override run_ocr()
# -----------------------------

def run_ocr(image_id: str) -> tuple[str, str]:
    img = load_image(image_id)
    if img is None:
        return "", ""

    img_proc = preprocess(img)
    img_np = np.array(img_proc)

    try:
        output = paddle_reader.predict(img_np)
    except AttributeError:
        output = paddle_reader.ocr(img_np)
    except Exception as e:
        print(f"OCR failed for {image_id}: {repr(e)}")
        return "", ""

    paddle_text, paddle_avg, paddle_lines, paddle_scores, polys = parse_paddle_detail(
        output, score_threshold=OCR_SCORE_THRESHOLD
    )

    final_text = paddle_text
    route = "paddle_only"

    if USE_VIETOCR_RESCUE and vietocr_reader is not None and not is_good_paddle_text(paddle_text, paddle_avg):
        viet_text = recognize_vietocr_on_polys(img_proc, polys)
        final_text, route = merge_paddle_vietocr(paddle_text, paddle_avg, viet_text)

    product_name = predict_product(final_text)
    product_name = apply_resnet_gate(product_name, image_id, img_proc)

    return final_text, product_name


# Smoke test after hybrid override.
first_id = test_df["image_id"].iloc[0]
ocr_text, product_name = run_ocr(first_id)
print("\nHybrid smoke test")
print("image_id    :", first_id)
print("ocr_text    :", ocr_text[:300] + ("..." if len(ocr_text) > 300 else ""))
print("product_name:", product_name or "(empty)")
print("VietOCR rescue:", "ON" if vietocr_reader is not None else "OFF")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
18533it [00:15, 1218.45it/s]


VietOCR rescue loaded on cpu

Hybrid smoke test
image_id    : img_2934
ocr_text    : THED:TIENTHANG-VTCNEWS AHH:TONG HOP HA LONG CANFOCO ISO 22000:2018 FSSC22000 pate TikTok @hanoinews.theanh28 HÀ NÔI vate NEWS NGAY DUNG: 11/01/2026 CHÍNH THĆC: NHÀ MÁY DÖ HŹP H LONG TAM DÙNG HOAT DÔNG TÙ 12/1
product_name: Ha Long Canfoco Pate
VietOCR rescue: ON


## 6.6) Optional ResNet visual gate

Cell này mặc định **tắt**. Nếu muốn thử visual gate, đổi `USE_RESNET_GATE = True`. ResNet chỉ chặn product khi domain ảnh mâu thuẫn rất chắc, không tự predict product toàn bộ.


In [11]:
# ============================================================
# OPTIONAL RESNET VISUAL DOMAIN GATE
# ============================================================
# Default OFF for final stability. Turn on only for an experiment after you can validate LB.
USE_RESNET_GATE = False
RESNET_EPOCHS = 2
RESNET_BATCH_SIZE = 32
RESNET_GATE_CONF = 0.92

# Classes: 0=no_product_or_noise, 1=pate, 2=milk
DOMAIN_ID_TO_NAME = {0: "none", 1: "pate", 2: "milk"}
DOMAIN_NAME_TO_ID = {v: k for k, v in DOMAIN_ID_TO_NAME.items()}

PATE_DOMAIN_KEYS = [
    "pate", "patê", "pa te", "ha long", "hạ long", "canfoco", "cột đèn", "cot den",
    "vissan", "đồ hộp", "do hop", "thịt hộp", "thit hop", "xúc xích", "xuc xich",
]
MILK_DOMAIN_KEYS = [
    "sữa", "sua", "milk", "nan", "nestle", "nestlé", "vinamilk", "aptamil", "ensure",
    "similac", "pediasure", "friso", "meiji", "dielac", "nutifood", "grow",
]


def infer_domain(text: str) -> str:
    low = "" if text is None else str(text).lower()
    p = sum(k in low for k in PATE_DOMAIN_KEYS)
    m = sum(k in low for k in MILK_DOMAIN_KEYS)
    if p > m and p > 0:
        return "pate"
    if m > p and m > 0:
        return "milk"
    return "none"


visual_model = None
visual_transform_ready = False

if USE_RESNET_GATE:
    import paddle
    import paddle.nn as nn
    import paddle.vision.transforms as T
    from paddle.io import Dataset, DataLoader
    from paddle.vision.models import resnet18

    def find_train_images_dir(input_dir: Path) -> Path | None:
        candidates = [
            "train_images/train_images/images", "train_images/images", "train_images",
            "images/train", "train/images",
        ]
        for rel in candidates:
            p = input_dir / rel
            if p.is_dir() and any(p.glob("*.jpg")):
                return p
        # fallback search
        for p in sorted(input_dir.rglob("*.jpg")):
            if "train" in str(p.parent).lower():
                return p.parent
        return None

    TRAIN_IMAGES_DIR = find_train_images_dir(INPUT_DIR)
    if TRAIN_IMAGES_DIR is None:
        print("[warn] Train image dir not found. ResNet gate disabled.")
        USE_RESNET_GATE = False
    else:
        train_rows = train_labels_df.copy()
        train_rows["image_id"] = train_rows["image_id"].astype(str).str.strip()
        train_rows["product_name"] = train_rows["product_name"].fillna("").astype(str)
        train_rows["domain"] = train_rows["product_name"].map(infer_domain)
        train_rows["label"] = train_rows["domain"].map(DOMAIN_NAME_TO_ID).fillna(0).astype("int64")

        # Keep all rows, but sample no_product to avoid it dominating.
        pos = train_rows[train_rows["label"] != 0]
        neg = train_rows[train_rows["label"] == 0]
        if len(pos) > 0 and len(neg) > len(pos):
            neg = neg.sample(n=len(pos), random_state=42)
        train_rows = pd.concat([pos, neg], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

        print("ResNet gate train rows:", len(train_rows))
        print(train_rows["domain"].value_counts())

        transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        class VisualDomainDataset(Dataset):
            def __init__(self, df, image_dir, transform):
                self.df = df.reset_index(drop=True)
                self.image_dir = Path(image_dir)
                self.transform = transform

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                image_id = row["image_id"]
                path = self.image_dir / f"{image_id}.jpg"
                img = Image.open(path).convert("RGB")
                x = self.transform(img)
                y = np.array(row["label"], dtype="int64")
                return x, y

        ds = VisualDomainDataset(train_rows, TRAIN_IMAGES_DIR, transform)
        dl = DataLoader(ds, batch_size=RESNET_BATCH_SIZE, shuffle=True, num_workers=2, drop_last=False)

        paddle.set_device("gpu:0" if paddle.device.is_compiled_with_cuda() else "cpu")
        visual_model = resnet18(pretrained=True, num_classes=3)
        criterion = nn.CrossEntropyLoss()
        opt = paddle.optimizer.AdamW(parameters=visual_model.parameters(), learning_rate=1e-4, weight_decay=1e-4)

        visual_model.train()
        for epoch in range(1, RESNET_EPOCHS + 1):
            total_loss, total, correct = 0.0, 0, 0
            for x, y in tqdm(dl, desc=f"ResNet gate epoch {epoch}/{RESNET_EPOCHS}"):
                logits = visual_model(x)
                loss = criterion(logits, y)
                loss.backward()
                opt.step()
                opt.clear_grad()
                total_loss += float(loss.numpy()) * len(y)
                pred = logits.argmax(axis=1)
                correct += int((pred == y).numpy().sum())
                total += len(y)
            print(f"epoch={epoch} loss={total_loss/max(1,total):.4f} acc={correct/max(1,total):.4f}")

        visual_model.eval()
        visual_transform_ready = True

        def predict_visual_domain_from_image(img: Image.Image):
            if visual_model is None or not visual_transform_ready:
                return "none", 0.0
            x = transform(img).unsqueeze(0)
            with paddle.no_grad():
                logits = visual_model(x)
                prob = paddle.nn.functional.softmax(logits, axis=1).numpy()[0]
            idx = int(prob.argmax())
            return DOMAIN_ID_TO_NAME[idx], float(prob[idx])

        def apply_resnet_gate(product_name: str, image_id: str, img: Image.Image) -> str:
            product_name = "" if product_name is None else str(product_name).strip()
            if not product_name:
                return ""
            prod_domain = infer_domain(product_name)
            if prod_domain == "none":
                return product_name
            img_domain, conf = predict_visual_domain_from_image(img)
            if conf >= RESNET_GATE_CONF and img_domain != "none" and img_domain != prod_domain:
                return ""
            return product_name

        print("ResNet visual gate enabled. Gate confidence:", RESNET_GATE_CONF)
else:
    print("ResNet visual gate is OFF. Set USE_RESNET_GATE=True to train/use it.")


ResNet visual gate is OFF. Set USE_RESNET_GATE=True to train/use it.


## 7) Preview OCR on a small slice



In [16]:
# Set PREVIEW_START / PREVIEW_N to inspect OCR quality before running full inference.
PREVIEW_START = 0
PREVIEW_N = 100

preview_ids = list(test_df["image_id"].iloc[PREVIEW_START:PREVIEW_START + PREVIEW_N])
preview_rows = []

print(f"Preview real PaddleOCR: start={PREVIEW_START}, n={PREVIEW_N}, device={OCR_DEVICE}")
print("=" * 90)

for k, image_id in enumerate(preview_ids, start=1):
    t0 = time.perf_counter()
    ocr_text, product_name = run_ocr(image_id)
    dt = time.perf_counter() - t0
    preview_rows.append({
        "image_id": image_id,
        "ocr_text": ocr_text,
        "product_name": product_name,
        "sec": round(dt, 2),
    })
    print(f"\n[{k:02d}/{len(preview_ids):02d}] {image_id} | {dt:.2f}s")
    print("  OCR    :", ocr_text[:220] + ("..." if len(ocr_text) > 220 else "") if ocr_text else "(empty)")
    print("  PRODUCT:", product_name or "(empty)")

preview_df = pd.DataFrame(preview_rows)
display(preview_df)



Preview real PaddleOCR: start=0, n=100, device=gpu

[01/100] img_2934 | 3.88s
  OCR    : THEOTIENTHANG-VICNEWS ÀNH:TONG HOP HA LONG CANFOCO ISO 22000:2018 FSSC 22000 oate TikTok @hanoinews.theanh28 HÀ NÔI pate NEWS NGAY DANG:11/01/2026 CHÍNH THÚC: NHÀ MÁY DÕ HŹP H LONG TAM DÙNG HOAT DÔNG TÙ 12/1
  PRODUCT: Ha Long Canfoco

[02/100] img_2935 | 0.23s
  OCR    : (empty)
  PRODUCT: (empty)

[03/100] img_2936 | 1.37s
  OCR    : Op News 11/01/2026 DO HOP HA LONG NFOCO TAM DUNG SAN ng XUÁT. Phòng, om.vn
  PRODUCT: Ha Long Canfoco

[04/100] img_2937 | 0.36s
  OCR    : SG A 0 A 0 O
  PRODUCT: (empty)

[05/100] img_2938 | 0.88s
  OCR    : Op News 11/01/2026 HALONG CANFOCO ng Phòng, om.vn
  PRODUCT: Ha Long Canfoco

[06/100] img_2939 | 1.34s
  OCR    : Op News 11/01/2026 DO HOP HA LONG eng TAM DU'NG SAN XUÁT. Phòng, om.vn
  PRODUCT: Đồ Hộp Hạ Long

[07/100] img_2940 | 1.55s
  OCR    : LÕI NHÁN NHU VÊ N.HÂN Q.UÀ VÀ DI SÁN TRONG K.INH D.OANH
  PRODUCT: (empty)

[08/100] img_2941 | 0.28s
  OCR    : 

,image_id,ocr_text,product_name,sec
0,img_2934,THEOTIENTHANG-VICNEWS ÀNH:TONG HOP HA LONG CAN...,Ha Long Canfoco,3.88
1,img_2935,,,0.23
2,img_2936,Op News 11/01/2026 DO HOP HA LONG NFOCO TAM DU...,Ha Long Canfoco,1.37
3,img_2937,SG A 0 A 0 O,,0.36
4,img_2938,"Op News 11/01/2026 HALONG CANFOCO ng Phòng, om.vn",Ha Long Canfoco,0.88
...,...,...,...,...
95,img_3029,Ông Truong Sý Toàn tng giám đc công ty c phn đ...,Đồ Hộp Hạ Long,3.69
96,img_3030,,,0.12
97,img_3031,Ông Truơng S Toàn tng giám đc công ty c phn đô...,Đồ Hộp Hạ Long,3.97
98,img_3032,/EON BáchhóaXANH COM Hat nèm Chinsu Thit Nuoc ...,Ha Long Canfoco Pate,4.49


## 8) Full OCR inference with checkpoint resume



In [13]:
# ============================================================
# Full inference config
# ============================================================
START_INDEX = 0       # set 700 nếu muốn chạy từ img index 700 trở đi
LIMIT = None          # None = full test set. Use e.g. 50 for a quick test.
SAVE_EVERY = 25
# Fresh judge run will not have checkpoint. Unique checkpoint name avoids reusing old LR/VietOCR outputs.
FORCE_REOCR = False   # True = ignore this notebook's no-LR checkpoint and OCR lại từ đầu

all_ids = list(test_df["image_id"].astype(str))
if LIMIT is None:
    target_ids = all_ids[START_INDEX:]
else:
    target_ids = all_ids[START_INDEX:START_INDEX + int(LIMIT)]

done_map = {}
if CHECKPOINT_CSV.exists() and not FORCE_REOCR:
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    if {"image_id", "ocr_text", "product_name"}.issubset(ckpt.columns):
        ckpt["image_id"] = ckpt["image_id"].astype(str).str.strip()
        ckpt = ckpt.drop_duplicates("image_id", keep="last")
        done_map = ckpt.set_index("image_id")[["ocr_text", "product_name"]].to_dict("index")
        print(f"Resuming from checkpoint: {len(done_map):,} rows")
    else:
        print("Checkpoint exists but columns are invalid; ignoring it.")

pending = [x for x in target_ids if x not in done_map]
print(f"Target : {len(target_ids):,} images | Done: {len(target_ids) - len(pending):,} | Pending: {len(pending):,}")
print(f"Output checkpoint: {CHECKPOINT_CSV}")

errors = 0
new_rows = []
run_start = time.perf_counter()

for idx, image_id in enumerate(tqdm(pending, desc="PaddleOCR real OCR"), start=1):
    t0 = time.perf_counter()
    ocr_text, product_name = run_ocr(image_id)
    elapsed = time.perf_counter() - t0

    if not str(ocr_text).strip() and image_path_for_id(image_id).exists():
        errors += 1

    row = {
        "image_id": image_id,
        "ocr_text": ocr_text,
        "product_name": product_name,
        "sec": round(elapsed, 3),
    }
    done_map[image_id] = {"ocr_text": ocr_text, "product_name": product_name}
    new_rows.append(row)

    if idx % SAVE_EVERY == 0:
        ckpt_rows = []
        # Preserve test order where possible.
        for img_id in all_ids:
            if img_id in done_map:
                ckpt_rows.append({"image_id": img_id, **done_map[img_id]})
        pd.DataFrame(ckpt_rows).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")
        avg = (time.perf_counter() - run_start) / idx
        print(f"Saved checkpoint at {idx:,}/{len(pending):,} pending | avg {avg:.2f}s/img | empty OCR in this run {errors:,}")

# Final checkpoint
ckpt_rows = []
for img_id in all_ids:
    if img_id in done_map:
        ckpt_rows.append({"image_id": img_id, **done_map[img_id]})
results_df = pd.DataFrame(ckpt_rows)
results_df.to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")

print("\nDone.")
print(f"Checkpoint rows: {len(results_df):,}")
if len(results_df):
    ocr_fill = (results_df["ocr_text"].fillna("").astype(str).str.strip() != "").mean()
    prod_fill = (results_df["product_name"].fillna("").astype(str).str.strip() != "").mean()
    print(f"OCR fill     : {ocr_fill:.1%}")
    print(f"Product fill : {prod_fill:.1%}")
    print(f"OCR failures this run: {errors:,}")
    display(results_df.head())



Target : 2,006 images | Done: 0 | Pending: 2,006
Output checkpoint: /kaggle/working/checkpoint_hybrid_paddle_vietocr_resnet.csv


PaddleOCR real OCR:   0%|          | 0/2006 [00:00<?, ?it/s]

Saved checkpoint at 25/2,006 pending | avg 1.03s/img | empty OCR in this run 5
Saved checkpoint at 50/2,006 pending | avg 1.19s/img | empty OCR in this run 9
Saved checkpoint at 75/2,006 pending | avg 1.35s/img | empty OCR in this run 11
Saved checkpoint at 100/2,006 pending | avg 1.37s/img | empty OCR in this run 15
Saved checkpoint at 125/2,006 pending | avg 1.64s/img | empty OCR in this run 18
Saved checkpoint at 150/2,006 pending | avg 1.70s/img | empty OCR in this run 19
Saved checkpoint at 175/2,006 pending | avg 1.68s/img | empty OCR in this run 23
Saved checkpoint at 200/2,006 pending | avg 1.85s/img | empty OCR in this run 25
Saved checkpoint at 225/2,006 pending | avg 2.00s/img | empty OCR in this run 30
Saved checkpoint at 250/2,006 pending | avg 2.03s/img | empty OCR in this run 33
Saved checkpoint at 275/2,006 pending | avg 1.99s/img | empty OCR in this run 35
Saved checkpoint at 300/2,006 pending | avg 2.02s/img | empty OCR in this run 38
Saved checkpoint at 325/2,006 pen

,image_id,ocr_text,product_name
0,img_2934,THED:TIENTHANG-VTCNEWS AHH:TONG HOP HA LONG CA...,Ha Long Canfoco Pate
1,img_2935,,
2,img_2936,Op News 11/01/2026 DO HOP HA LONG NFOCO TAM DÜ...,Ha Long Canfoco
3,img_2937,ST IDO A 0 DS,
4,img_2938,"Op News 11/01/2026 HALONG CANFOCO ng Phòng, om.vn",Ha Long Canfoco


## 9) Validate + export `submission.csv`



In [14]:
import csv
import re
import unicodedata

# ============================================================
# FINAL SUBMISSION BUILD + OCR JUNK GUARD
# ============================================================
# Mục tiêu:
# - Giữ nguyên các cụm OCR Latin dù sai dấu/sai chính tả:
#   ví dụ DÔ HÔP HA LONG, COTDEN, HAIPHONG...
# - Chỉ xoá ký tự rác không hữu ích:
#   chữ Hán/Nhật/Hàn, emoji, sao, bullet, ký hiệu lạ...
# - Blank các OCR cực ngắn/vô nghĩa như 心, 福, ★, O, Y, 5, CS, 24...
# - Không canonical kiểu DÔ HÔP HA LONG -> ĐỒ HỘP HẠ LONG

OCR_ALLOWED_PUNCT = set(".,:;!?/\\-–—_+&%@#()[]{}'\"°²³=")

OCR_TRANSLATE_CHARS = str.maketrans({
    "“": '"',
    "”": '"',
    "„": '"',
    "‟": '"',
    "‘": "'",
    "’": "'",
    "‚": "'",
    "‛": "'",
    "…": "...",
    "×": "x",
})


def _is_latin_letter(ch: str) -> bool:
    return ch.isalpha() and "LATIN" in unicodedata.name(ch, "")


def _useful_latin_digit_count(text: str) -> int:
    return sum(
        1
        for ch in str(text)
        if ch.isdigit() or _is_latin_letter(ch)
    )


def clean_ocr_visible_text(x) -> str:
    """
    Cleaner giống bản CSV lọc rác tốt hơn:
    - Giữ Latin/Vietnamese/digit/useful punctuation.
    - Xoá CJK/Japanese/Korean/emoji/symbol lạ.
    - Nếu còn <=2 ký tự chữ/số hữu ích thì xem là OCR rác và blank.
    """
    x = "" if x is None or pd.isna(x) else str(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = unicodedata.normalize("NFC", x).translate(OCR_TRANSLATE_CHARS)

    out = []
    for ch in x:
        cat = unicodedata.category(ch)

        if _is_latin_letter(ch) or ch.isdigit():
            out.append(ch)
        elif cat.startswith("M"):
            # Giữ combining marks để không làm hỏng tiếng Việt dạng decomposed.
            out.append(ch)
        elif ch.isspace():
            out.append(" ")
        elif ch in OCR_ALLOWED_PUNCT:
            out.append(ch)
        # Còn lại bỏ: Hán/Nhật/Hàn, emoji, sao, ký hiệu lạ...

    cleaned = "".join(out)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Nếu chỉ còn dấu câu/ký hiệu thì blank.
    if not any(ch.isdigit() or _is_latin_letter(ch) for ch in cleaned):
        return " "

    # Blank OCR quá ngắn/vô nghĩa như O, Y, 5, CS, 24...
    if _useful_latin_digit_count(cleaned) <= 2:
        return " "

    return cleaned if cleaned else " "


def clean_submission_cell(x) -> str:
    """
    Cleaner nhẹ cho product_name hoặc field thường.
    Không xoá Unicode ở product_name để tránh làm mất dấu sản phẩm.
    """
    x = "" if x is None or pd.isna(x) else str(x)
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else " "


# Build submission from checkpoint/results.
# Missing rows are filled with a single space to keep Kaggle format valid.
if "results_df" not in globals() or results_df is None or len(results_df) == 0:
    if CHECKPOINT_CSV.exists():
        results_df = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    else:
        raise RuntimeError("Chưa có results_df/checkpoint. Hãy chạy full inference trước.")

sub = sample_df[["image_id"]].copy()
sub["image_id"] = sub["image_id"].astype(str).str.strip()

r = results_df[["image_id", "ocr_text", "product_name"]].copy()
r["image_id"] = r["image_id"].astype(str).str.strip()
r = r.drop_duplicates("image_id", keep="last")

sub = sub.merge(r, on="image_id", how="left")

# Quan trọng: chỉ ocr_text dùng junk guard mạnh.
sub["ocr_text"] = sub["ocr_text"].map(clean_ocr_visible_text)

# product_name chỉ clean khoảng trắng/newline.
sub["product_name"] = sub["product_name"].map(clean_submission_cell)

# Nếu muốn bản OCR-only như mấy file trước, mở comment dòng dưới:
# sub["product_name"] = " "

sub = sub[["image_id", "ocr_text", "product_name"]]

checks = {}
checks["AC-1 Row count match"] = len(sub) == len(sample_df)
checks["AC-2 No duplicate IDs"] = not sub["image_id"].duplicated().any()
checks["AC-3 No null values"] = not sub.isnull().any().any()
checks["AC-4 Columns correct"] = list(sub.columns) == ["image_id", "ocr_text", "product_name"]
checks["AC-5 No newline/tab"] = not sub["ocr_text"].astype(str).str.contains(r"\n|\t", regex=True, na=False).any()
checks["AC-6 IDs match sample"] = set(sub["image_id"]) == set(sample_df["image_id"].astype(str))

all_pass = True
print("Validating submission format...")
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    all_pass &= bool(ok)

n_missing_ocr = int((sub["ocr_text"].str.strip() == "").sum() + (sub["ocr_text"] == " ").sum())
n_missing_product = int((sub["product_name"].str.strip() == "").sum() + (sub["product_name"] == " ").sum())

# Đếm nhanh còn dòng OCR có ký tự ngoài Latin/số/dấu hữu ích hay không.
def has_bad_ocr_char(s: str) -> bool:
    for ch in str(s):
        if ch.isspace() or ch.isdigit() or ch in OCR_ALLOWED_PUNCT:
            continue
        if _is_latin_letter(ch):
            continue
        if unicodedata.category(ch).startswith("M"):
            continue
        return True
    return False

n_bad_chars_left = int(sub["ocr_text"].map(has_bad_ocr_char).sum())

print(f"\nRows               : {len(sub):,}")
print(f"Blank OCR cells    : {n_missing_ocr:,}")
print(f"Blank product cells: {n_missing_product:,}")
print(f"Bad-char OCR rows  : {n_bad_chars_left:,}")

if all_pass:
    sub.to_csv(OUTPUT_CSV, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"\nSaved: {OUTPUT_CSV}")
    display(sub.head())

    try:
        from IPython.display import FileLink, display
        display(FileLink(str(OUTPUT_CSV)))
    except Exception:
        pass
else:
    print("\nValidation failed. Fix checks above before submitting.")

Validating submission format...
  [PASS] AC-1 Row count match
  [PASS] AC-2 No duplicate IDs
  [PASS] AC-3 No null values
  [PASS] AC-4 Columns correct
  [PASS] AC-5 No newline/tab
  [PASS] AC-6 IDs match sample

Rows               : 2,006
Blank OCR cells    : 578
Blank product cells: 1,680
Bad-char OCR rows  : 0

Saved: /kaggle/working/submission.csv


,image_id,ocr_text,product_name
0,img_2934,THED:TIENTHANG-VTCNEWS AHH:TONG HOP HA LONG CA...,Ha Long Canfoco Pate
1,img_2935,,
2,img_2936,Op News 11/01/2026 DO HOP HA LONG NFOCO TAM DÜ...,Ha Long Canfoco
3,img_2937,ST IDO A 0 DS,
4,img_2938,"Op News 11/01/2026 HALONG CANFOCO ng Phòng, om.vn",Ha Long Canfoco


/kaggle/working/submission.csv

## 10) Debug helpers



In [ ]:
print("=== /kaggle/working files ===")
for p in sorted(WORK_DIR.glob("*")):
    if p.is_file():
        print(p, p.stat().st_size, "bytes")

print("\n=== Key variables ===")
for name in [
    "INPUT_DIR", "IMAGES_DIR", "TEST_CSV", "SAMPLE_CSV", "CHECKPOINT_CSV", "OUTPUT_CSV",
    "OCR_DEVICE", "OCR_SCORE_THRESHOLD", "MAX_DIM", "product_predictor",
]:
    print(name, "=>", globals().get(name, None))

if CHECKPOINT_CSV.exists():
    ck = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    print("\ncheckpoint shape:", ck.shape)
    display(ck.tail())

if OUTPUT_CSV.exists():
    ss = pd.read_csv(OUTPUT_CSV, keep_default_na=False)
    print("\nsubmission shape:", ss.shape)
    display(ss.head())



## Notes

- Đây là nhánh thử nghiệm bảo thủ: **PaddleOCR base + VietOCR rescue only + product no-LR**.
- VietOCR không thay toàn bộ OCR; chỉ cứu khi Paddle blank/yếu và text VietOCR vượt quality gate.
- ResNet visual gate mặc định tắt. Bật `USE_RESNET_GATE=True` nếu muốn train gate domain ảnh, nhưng cần submit test riêng trước khi dùng final.
- Nếu VietOCR vẫn làm nhiễu, tăng `VIETOCR_MIN_QUALITY` lên `0.80` hoặc tắt `USE_VIETOCR_RESCUE=False`.
- Nếu muốn quay về bản final ổn định nhất đã test, dùng notebook `ura_final_stable_paddleocr_gpu_no_lr.ipynb`.
- File nộp chính thức nằm ở `/kaggle/working/submission.csv`.


In [18]:
# ============================================================
# Install PyTorch CPU + Transformers deps for Flan-T5
# ============================================================
# Chạy cell này trước cell Flan-T5.
# Sau khi chạy xong, restart runtime/kernel rồi chạy lại notebook/cell Flan-T5.

import sys
import subprocess
import importlib.util

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print("Before install:")
print("torch       :", has_module("torch"))
print("transformers:", has_module("transformers"))
print("sentencepiece:", has_module("sentencepiece"))
print("accelerate :", has_module("accelerate"))

# Cài torch CPU để tránh đụng CUDA/NCCL với PaddleOCR.
# Flan-T5-small chạy CPU vẫn thử nghiệm được, chỉ chậm hơn GPU.
if not has_module("torch"):
    print("\nInstalling CPU PyTorch...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch",
        "--index-url", "https://download.pytorch.org/whl/cpu",
    ])

# Cài các dependency cho Flan-T5.
missing = []
for pkg, mod in [
    ("transformers", "transformers"),
    ("sentencepiece", "sentencepiece"),
    ("accelerate", "accelerate"),
    ("safetensors", "safetensors"),
]:
    if not has_module(mod):
        missing.append(pkg)

if missing:
    print("\nInstalling:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *missing,
    ])

print("\nAfter install check:")
import torch
import transformers
import sentencepiece

print("torch version       :", torch.__version__)
print("torch cuda available:", torch.cuda.is_available())
print("transformers version:", transformers.__version__)

print("\nDONE. Bây giờ restart runtime/kernel rồi chạy lại cell Flan-T5.")

Before install:
torch       : True
transformers: True
sentencepiece: True
accelerate : True

After install check:
torch version       : 2.12.1+cpu
torch cuda available: False
transformers version: 5.0.0

DONE. Bây giờ restart runtime/kernel rồi chạy lại cell Flan-T5.
